In [1]:
import os
from pathlib import Path
import json
import numpy as np

# Install if needed:
# pip install rasterio pillow matplotlib

import rasterio
from rasterio.plot import reshape_as_image
from PIL import Image


# ============================================================
# 1. ADD YOUR TIFF PATHS HERE
# ============================================================
tiff_files = [
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_season_ndvi_2022_2023_500m.tif",
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_late_season_ndvi_2022_2023_500m.tif",
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif",
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_publication_map_source_2022_2023_500m.tif",
]

# ============================================================
# 2. OUTPUT FOLDER
# ============================================================
output_dir = Path(r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

metadata_report = []


def normalize_band(arr, nodata=None):
    """Normalize a single band to 0-255 for PNG export."""
    arr = arr.astype("float32")

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    valid = np.isfinite(arr)
    if not np.any(valid):
        return np.zeros(arr.shape, dtype=np.uint8)

    min_val = np.nanmin(arr)
    max_val = np.nanmax(arr)

    if max_val == min_val:
        out = np.zeros(arr.shape, dtype=np.uint8)
        out[valid] = 128
        return out

    scaled = (arr - min_val) / (max_val - min_val)
    scaled = np.clip(scaled, 0, 1)
    scaled = (scaled * 255).astype(np.uint8)
    scaled[~valid] = 0
    return scaled


def band_stats(arr, nodata=None):
    """Compute summary statistics for a band."""
    arr = arr.astype("float64")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    valid = np.isfinite(arr)
    if not np.any(valid):
        return {
            "min": None,
            "max": None,
            "mean": None,
            "std": None,
            "valid_pixel_count": 0
        }

    return {
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "std": float(np.nanstd(arr)),
        "valid_pixel_count": int(np.sum(valid))
    }


for tif_path in tiff_files:
    tif_path = Path(tif_path)

    if not tif_path.exists():
        print(f"File not found: {tif_path}")
        continue

    print("=" * 90)
    print(f"READING: {tif_path}")
    print("=" * 90)

    with rasterio.open(tif_path) as src:
        meta = {
            "file_name": tif_path.name,
            "full_path": str(tif_path),
            "driver": src.driver,
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "crs": str(src.crs),
            "transform": str(src.transform),
            "bounds": {
                "left": src.bounds.left,
                "bottom": src.bounds.bottom,
                "right": src.bounds.right,
                "top": src.bounds.top,
            },
            "dtype": src.dtypes,
            "nodata": src.nodata,
            "descriptions": src.descriptions,
            "tags": src.tags(),
            "band_stats": []
        }

        print(f"Driver       : {src.driver}")
        print(f"Size         : {src.width} x {src.height}")
        print(f"Band count   : {src.count}")
        print(f"CRS          : {src.crs}")
        print(f"Transform    : {src.transform}")
        print(f"Bounds       : {src.bounds}")
        print(f"NoData       : {src.nodata}")
        print(f"Dtypes       : {src.dtypes}")
        print(f"Descriptions : {src.descriptions}")
        print(f"Tags         : {src.tags()}")

        # ---------- Read all bands ----------
        data = src.read()

        # ---------- Per-band stats ----------
        for b in range(1, src.count + 1):
            arr = src.read(b)
            stats = band_stats(arr, src.nodata)
            meta["band_stats"].append({
                "band": b,
                "description": src.descriptions[b - 1] if src.descriptions else None,
                **stats
            })

            print(f"\nBand {b}:")
            print(f"  Description      : {src.descriptions[b - 1] if src.descriptions else None}")
            print(f"  Min              : {stats['min']}")
            print(f"  Max              : {stats['max']}")
            print(f"  Mean             : {stats['mean']}")
            print(f"  Std              : {stats['std']}")
            print(f"  Valid pixel count: {stats['valid_pixel_count']}")

        # ========================================================
        # PNG EXPORT
        # ========================================================
        png_path = output_dir / (tif_path.stem + ".png")

        if src.count == 1:
            # Single-band TIFF
            band = src.read(1)
            band_8bit = normalize_band(band, src.nodata)
            img = Image.fromarray(band_8bit)
            img.save(png_path)

        elif src.count >= 3:
            # Multi-band TIFF: take first 3 bands as RGB
            r = normalize_band(src.read(1), src.nodata)
            g = normalize_band(src.read(2), src.nodata)
            b = normalize_band(src.read(3), src.nodata)
            rgb = np.dstack([r, g, b])
            img = Image.fromarray(rgb)
            img.save(png_path)

        else:
            # Two-band TIFF: export first band only
            band = src.read(1)
            band_8bit = normalize_band(band, src.nodata)
            img = Image.fromarray(band_8bit)
            img.save(png_path)

        print(f"\nPNG saved to: {png_path}")

        metadata_report.append(meta)

# ============================================================
# 3. SAVE METADATA REPORT
# ============================================================
report_path = output_dir / "tiff_metadata_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(metadata_report, f, indent=2)

print("\n" + "=" * 90)
print("PROCESS COMPLETE")
print("=" * 90)
print(f"PNG outputs folder : {output_dir}")
print(f"Metadata report    : {report_path}")

READING: D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_season_ndvi_2022_2023_500m.tif
Driver       : GTiff
Size         : 1957 x 1292
Band count   : 3
CRS          : EPSG:4326
Transform    : | 0.00, 0.00, 74.03|
| 0.00,-0.00, 26.87|
| 0.00, 0.00, 1.00|
Bounds       : BoundingBox(left=74.02567098786918, bottom=21.069984989023375, right=82.8156860429787, top=26.873101724435486)
NoData       : None
Dtypes       : ('uint8', 'uint8', 'uint8')
Descriptions : ('vis-red', 'vis-green', 'vis-blue')
Tags         : {'AREA_OR_POINT': 'Area'}

Band 1:
  Description      : vis-red
  Min              : 0.0
  Max              : 246.0
  Mean             : 170.10341696315996
  Std              : 95.53228728958771
  Valid pixel count: 2528444

Band 2:
  Description      : vis-green
  Min              : 0.0
  Max              : 234.0
  Mean             : 172.46506388909543
  Std              : 93.78452043529664
  Valid pixel count: 2528444

Band 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10288\4208467427.py:54: RuntimeWarning: invalid value encountered in cast
  scaled = (scaled * 255).astype(np.uint8)



PNG saved to: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\mp_wheat_s2_publication_map_source_2022_2023_500m.png

PROCESS COMPLETE
PNG outputs folder : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs
Metadata report    : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\tiff_metadata_report.json


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import TwoSlopeNorm


# ============================================================
# INPUT AND OUTPUT PATHS
# ============================================================

source_tif = Path(
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2"
    r"\MP_Wheat_Sentinel2_Publication_Maps"
    r"\mp_wheat_s2_publication_map_source_2022_2023_500m.tif"
)

output_dir = Path(
    r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

figure_11_path = output_dir / (
    "Figure_11_Early_to_Peak_NDVI_Change_2022_2023.png"
)

figure_12_path = output_dir / (
    "Figure_12_Peak_to_Late_NDVI_Change_2022_2023.png"
)


# ============================================================
# READ BANDS 4 AND 5
# ============================================================

with rasterio.open(source_tif) as src:
    early_to_peak = src.read(4, masked=True).astype("float64")
    peak_to_late = src.read(5, masked=True).astype("float64")

    bounds = src.bounds
    crs = src.crs
    band_descriptions = src.descriptions


print("Source file:", source_tif)
print("CRS:", crs)
print("Bounds:", bounds)
print("Band descriptions:", band_descriptions)


# ============================================================
# REMOVE INVALID VALUES
# ============================================================

early_to_peak = np.ma.masked_invalid(early_to_peak)
peak_to_late = np.ma.masked_invalid(peak_to_late)

# Mask improbable values if required.
# NDVI differences normally remain approximately between -2 and +2.
early_to_peak = np.ma.masked_where(
    (early_to_peak < -2) | (early_to_peak > 2),
    early_to_peak
)

peak_to_late = np.ma.masked_where(
    (peak_to_late < -2) | (peak_to_late > 2),
    peak_to_late
)


# ============================================================
# USE A COMMON ZERO-CENTRED SCALE
# ============================================================

combined_values = np.concatenate([
    early_to_peak.compressed(),
    peak_to_late.compressed()
])

# Robust limits reduce the influence of isolated extreme pixels.
lower_percentile = np.percentile(combined_values, 2)
upper_percentile = np.percentile(combined_values, 98)

common_limit = max(
    abs(lower_percentile),
    abs(upper_percentile)
)

norm = TwoSlopeNorm(
    vmin=-common_limit,
    vcenter=0.0,
    vmax=common_limit
)

extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top
]


# ============================================================
# FUNCTION FOR EXPORTING A CHANGE MAP
# ============================================================

def export_change_map(
    raster_array,
    output_path,
    map_title
):
    fig, ax = plt.subplots(figsize=(10, 7))

    image = ax.imshow(
        raster_array,
        extent=extent,
        origin="upper",
        cmap="RdYlGn",
        norm=norm,
        interpolation="nearest"
    )

    ax.set_title(
        map_title,
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)

    ax.tick_params(
        axis="both",
        labelsize=9
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        orientation="vertical",
        fraction=0.035,
        pad=0.025
    )

    colorbar.set_label(
        "NDVI change",
        fontsize=11
    )

    colorbar.ax.tick_params(labelsize=9)

    ax.grid(
        visible=True,
        linewidth=0.4,
        alpha=0.35
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    print("PNG saved:", output_path)


# ============================================================
# FIGURE 11
# ============================================================

export_change_map(
    raster_array=early_to_peak,
    output_path=figure_11_path,
    map_title="Early-to-Peak NDVI Change, 2022–2023"
)


# ============================================================
# FIGURE 12
# ============================================================

export_change_map(
    raster_array=peak_to_late,
    output_path=figure_12_path,
    map_title="Peak-to-Late NDVI Change, 2022–2023"
)


# ============================================================
# PRINT SUMMARY STATISTICS
# ============================================================

def print_statistics(name, raster_array):
    values = raster_array.compressed()

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print("Valid pixels :", values.size)
    print("Minimum      :", float(np.min(values)))
    print("Maximum      :", float(np.max(values)))
    print("Mean         :", float(np.mean(values)))
    print("Median       :", float(np.median(values)))
    print("Std. dev.    :", float(np.std(values)))
    print("25th pct.    :", float(np.percentile(values, 25)))
    print("75th pct.    :", float(np.percentile(values, 75)))


print_statistics(
    "Figure 11: Early-to-Peak NDVI Change",
    early_to_peak
)

print_statistics(
    "Figure 12: Peak-to-Late NDVI Change",
    peak_to_late
)

print("\nPROCESS COMPLETE")

Source file: D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_publication_map_source_2022_2023_500m.tif
CRS: EPSG:4326
Bounds: BoundingBox(left=74.02567098786918, bottom=21.069984989023375, right=82.8156860429787, top=26.873101724435486)
Band descriptions: ('early_ndvi', 'peak_ndvi', 'late_ndvi', 'early_to_peak_ndvi', 'peak_to_late_ndvi')
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023.png
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023.png

Figure 11: Early-to-Peak NDVI Change
Valid pixels : 1991262
Minimum      : -0.822824239730835
Maximum      : 1.1344298124313354
Mean         : 0.1383379672330966
Median       : 0.11516956612467766
Std. dev.    : 0.20972000374808514
25th pct.    : -0.041853828355669975
75th pct.    : 0.3058997914195061

Figure 12: Peak-to-Late NDVI Change
Valid pixels : 199126

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import TwoSlopeNorm


# ============================================================
# INPUT FILE
# ============================================================

source_tif = Path(
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2"
    r"\MP_Wheat_Sentinel2_Publication_Maps"
    r"\mp_wheat_s2_publication_map_source_2022_2023_500m.tif"
)


# ============================================================
# OUTPUT FOLDER
# ============================================================

output_dir = Path(
    r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)


figure_11_path = output_dir / (
    "Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png"
)

figure_12_path = output_dir / (
    "Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.png"
)


# ============================================================
# READ BANDS 4 AND 5
# ============================================================

with rasterio.open(source_tif) as src:

    early_to_peak = src.read(
        4,
        masked=True
    ).astype("float64")

    peak_to_late = src.read(
        5,
        masked=True
    ).astype("float64")

    bounds = src.bounds
    crs = src.crs
    descriptions = src.descriptions


print("Source file:", source_tif)
print("CRS:", crs)
print("Band descriptions:", descriptions)


# ============================================================
# MASK INVALID VALUES
# ============================================================

early_to_peak = np.ma.masked_invalid(
    early_to_peak
)

peak_to_late = np.ma.masked_invalid(
    peak_to_late
)


# Retain only physically plausible NDVI-difference values
early_to_peak = np.ma.masked_where(
    (early_to_peak < -2.0) |
    (early_to_peak > 2.0),
    early_to_peak
)

peak_to_late = np.ma.masked_where(
    (peak_to_late < -2.0) |
    (peak_to_late > 2.0),
    peak_to_late
)


# ============================================================
# COMMON ZERO-CENTRED COLOUR SCALE
# ============================================================

combined_values = np.concatenate(
    [
        early_to_peak.compressed(),
        peak_to_late.compressed()
    ]
)

lower_limit = np.percentile(
    combined_values,
    2
)

upper_limit = np.percentile(
    combined_values,
    98
)

common_limit = max(
    abs(lower_limit),
    abs(upper_limit)
)

norm = TwoSlopeNorm(
    vmin=-common_limit,
    vcenter=0.0,
    vmax=common_limit
)


extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top
]


# ============================================================
# EXPORT FUNCTION WITHOUT TITLE OR CAPTION
# ============================================================

def export_change_map(
    raster_array,
    output_path
):

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    image = ax.imshow(
        raster_array,
        extent=extent,
        origin="upper",
        cmap="RdYlGn",
        norm=norm,
        interpolation="nearest"
    )

    # No embedded title or caption
    ax.set_title("")

    ax.set_xlabel(
        "Longitude",
        fontsize=11
    )

    ax.set_ylabel(
        "Latitude",
        fontsize=11
    )

    ax.tick_params(
        axis="both",
        labelsize=9
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        orientation="vertical",
        fraction=0.035,
        pad=0.025
    )

    colorbar.set_label(
        "NDVI change",
        fontsize=11
    )

    colorbar.ax.tick_params(
        labelsize=9
    )

    ax.grid(
        visible=True,
        linewidth=0.4,
        alpha=0.30
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False
    )

    plt.close(fig)

    print("PNG saved:", output_path)


# ============================================================
# CREATE FIGURE 11
# ============================================================

export_change_map(
    raster_array=early_to_peak,
    output_path=figure_11_path
)


# ============================================================
# CREATE FIGURE 12
# ============================================================

export_change_map(
    raster_array=peak_to_late,
    output_path=figure_12_path
)


print("\nPROCESS COMPLETE")
print("Figure 11:", figure_11_path)
print("Figure 12:", figure_12_path)

Source file: D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_publication_map_source_2022_2023_500m.tif
CRS: EPSG:4326
Band descriptions: ('early_ndvi', 'peak_ndvi', 'late_ndvi', 'early_to_peak_ndvi', 'peak_to_late_ndvi')
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.png

PROCESS COMPLETE
Figure 11: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png
Figure 12: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.png


In [6]:
import os
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# =============================================================================
# PATHS
# =============================================================================
source_tif = r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_publication_map_source_2022_2023_500m.tif"

output_dir = Path(r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def read_band_with_metadata(tif_path, band_number):
    """
    Read a single band from a GeoTIFF and return the array plus metadata.
    Values <= -0.99 are treated as invalid for these NDVI-based layers.
    """
    with rasterio.open(tif_path) as src:
        arr = src.read(band_number).astype(np.float32)
        bounds = src.bounds
        crs = src.crs
        descriptions = src.descriptions

    # Mask known invalid values
    arr[arr <= -0.99] = np.nan
    return arr, bounds, crs, descriptions


def save_single_band_map(
    arr,
    bounds,
    out_png,
    out_pdf,
    title,
    colorbar_label,
    cmap,
    vmin=None,
    vmax=None,
    center_zero=False,
    dpi=1000
):
    """
    Save a publication-quality map with axes and colorbar.
    """
    fig, ax = plt.subplots(figsize=(10, 7), facecolor="white")

    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

    if center_zero:
        valid = arr[np.isfinite(arr)]
        if valid.size == 0:
            raise ValueError("No finite values found in the raster band.")
        q02, q98 = np.nanpercentile(valid, [2, 98])
        lim = max(abs(q02), abs(q98))
        norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper",
            cmap=cmap,
            norm=norm
        )
    else:
        if vmin is None or vmax is None:
            valid = arr[np.isfinite(arr)]
            if valid.size == 0:
                raise ValueError("No finite values found in the raster band.")
            vmin, vmax = np.nanpercentile(valid, [2, 98])

        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )

    # Axes formatting
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
    ax.set_xlabel("Longitude (°E)", fontsize=11)
    ax.set_ylabel("Latitude (°N)", fontsize=11)

    ax.tick_params(axis="both", labelsize=10)
    ax.grid(False)

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(colorbar_label, fontsize=11)
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout()

    fig.savefig(out_png, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# =============================================================================
# FIGURE 10: PEAK-GROWTH NDVI
# Band descriptions:
# 1 = early_ndvi
# 2 = peak_ndvi
# 3 = late_ndvi
# 4 = early_to_peak_ndvi
# 5 = peak_to_late_ndvi
# =============================================================================
peak_arr, bounds, crs, descriptions = read_band_with_metadata(source_tif, 2)

fig10_png = output_dir / "Figure_10_Peak_Growth_NDVI_2022_2023.png"
fig10_pdf = output_dir / "Figure_10_Peak_Growth_NDVI_2022_2023.pdf"

save_single_band_map(
    arr=peak_arr,
    bounds=bounds,
    out_png=fig10_png,
    out_pdf=fig10_pdf,
    title="Peak-Growth NDVI Distribution in Madhya Pradesh Wheat Areas (2022-2023)",
    colorbar_label="Peak-growth NDVI",
    cmap="YlGn",
    vmin=0.0,
    vmax=0.9,
    center_zero=False,
    dpi=1000
)

# =============================================================================
# FIGURE 11: EARLY-TO-PEAK NDVI CHANGE
# =============================================================================
ep_arr, bounds, crs, descriptions = read_band_with_metadata(source_tif, 4)

fig11_png = output_dir / "Figure_11_Early_to_Peak_NDVI_Change_2022_2023.png"
fig11_pdf = output_dir / "Figure_11_Early_to_Peak_NDVI_Change_2022_2023.pdf"

save_single_band_map(
    arr=ep_arr,
    bounds=bounds,
    out_png=fig11_png,
    out_pdf=fig11_pdf,
    title="Early-to-Peak NDVI Change in Madhya Pradesh Wheat Areas (2022-2023)",
    colorbar_label="NDVI change (peak - early)",
    cmap="BrBG",
    center_zero=True,
    dpi=1000
)

# =============================================================================
# FIGURE 12: PEAK-TO-LATE NDVI CHANGE
# =============================================================================
pl_arr, bounds, crs, descriptions = read_band_with_metadata(source_tif, 5)

fig12_png = output_dir / "Figure_12_Peak_to_Late_NDVI_Change_2022_2023.png"
fig12_pdf = output_dir / "Figure_12_Peak_to_Late_NDVI_Change_2022_2023.pdf"

save_single_band_map(
    arr=pl_arr,
    bounds=bounds,
    out_png=fig12_png,
    out_pdf=fig12_pdf,
    title="Peak-to-Late NDVI Change in Madhya Pradesh Wheat Areas (2022-2023)",
    colorbar_label="NDVI change (late - peak)",
    cmap="BrBG",
    center_zero=True,
    dpi=1000
)

# =============================================================================
# REPORT
# =============================================================================
print("CRS:", crs)
print("Band descriptions:", descriptions)
print()
print("Saved files:")
print(fig10_png)
print(fig10_pdf)
print(fig11_png)
print(fig11_pdf)
print(fig12_png)
print(fig12_pdf)

CRS: EPSG:4326
Band descriptions: ('early_ndvi', 'peak_ndvi', 'late_ndvi', 'early_to_peak_ndvi', 'peak_to_late_ndvi')

Saved files:
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023.png
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023.pdf
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023.png
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023.pdf
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023.png
D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023.pdf


In [3]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

rgb_tifs = [
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_season_ndvi_2022_2023_500m.tif",
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif",
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_late_season_ndvi_2022_2023_500m.tif",
]

output_dir = Path(r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

for tif_path in rgb_tifs:
    tif_path = Path(tif_path)

    with rasterio.open(tif_path) as src:
        rgb = src.read([1, 2, 3]).astype(np.uint8)   # shape: (3, rows, cols)

    rgb = np.transpose(rgb, (1, 2, 0))  # to (rows, cols, 3)

    out_png = output_dir / f"{tif_path.stem}.png"

    plt.figure(figsize=(10, 7), facecolor="white")
    plt.imshow(rgb)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0, facecolor="white")
    plt.close()

    print("PNG saved:", out_png)

PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\mp_wheat_s2_early_season_ndvi_2022_2023_500m.png
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.png
PNG saved: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\mp_wheat_s2_late_season_ndvi_2022_2023_500m.png


In [7]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# PROJECT PATHS
# =============================================================================
PROJECT_ROOT = Path(r"D:\MP_Wheat_Project")
OUTPUT_DIR = PROJECT_ROOT / "publication_outputs" / "updated_blue_figures_6_to_9"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def find_file(root: Path, filename: str):
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find: {filename} under {root}")
    return matches[0]

def first_existing_column(df: pd.DataFrame, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"None of these columns were found: {candidates}")

def save_figure(fig, png_path: Path, pdf_path: Path):
    fig.savefig(png_path, dpi=1200, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def style_axis(ax):
    ax.set_axisbelow(True)
    ax.grid(
        True,
        which="major",
        axis="y",
        linestyle="--",
        linewidth=0.6,
        color="#D6E4F0",
        alpha=0.9,
        zorder=0
    )
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#7F9DB9")
    ax.spines["bottom"].set_color("#7F9DB9")
    ax.tick_params(axis="both", labelsize=10)

MODEL_NAME_MAP = {
    "baseline__prior_2year_moving_average": "Prior 2-year\nmoving average",
    "linear__elastic_net": "Elastic Net",
    "deep_learning__ft_transformer": "FT-Transformer",
    "deep_learning__lstm": "LSTM",
    "deep_learning__simple_rnn": "Simple RNN"
}

MODEL_ORDER = [
    "Prior 2-year\nmoving average",
    "Elastic Net",
    "FT-Transformer",
    "LSTM",
    "Simple RNN"
]

BLUE_2 = ["#CFE2F3", "#4F81BD"]
BLUE_3 = ["#DCEAF7", "#8DB4E2", "#2F75B5"]
BLUE_5 = ["#DCEAF7", "#B7D1EE", "#8DB4E2", "#5B9BD5", "#2F75B5"]

# =============================================================================
# LOCATE REQUIRED SOURCE FILES
# =============================================================================
iqr_contrast_file = find_file(PROJECT_ROOT, "outlier_sensitivity_iqr_contrasts.csv")
tail_metrics_file = find_file(PROJECT_ROOT, "outlier_sensitivity_tail_model_metrics.csv")
sequence_metrics_file = find_file(PROJECT_ROOT, "sequence_length_metric_summary.csv")

print("Found files:")
print("  IQR contrasts       :", iqr_contrast_file)
print("  Tail metrics        :", tail_metrics_file)
print("  Sequence metrics    :", sequence_metrics_file)

# =============================================================================
# FIGURE 6
# High-IQR Outlier Sensitivity
# =============================================================================
iqr_df = pd.read_csv(iqr_contrast_file)

# expected columns:
# protocol, model_display_name, non_outlier, high_iqr_outlier, ...
required_cols = ["protocol", "model_display_name", "non_outlier", "high_iqr_outlier"]
missing_cols = [c for c in required_cols if c not in iqr_df.columns]
if missing_cols:
    raise KeyError(f"Figure 6 source is missing columns: {missing_cols}")

protocol_order = ["district_grouped", "rolling_origin"]
protocol_title_map = {
    "district_grouped": "District-grouped validation",
    "rolling_origin": "Rolling-origin validation"
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="white", sharey=True)

for ax, protocol in zip(axes, protocol_order):
    sub = iqr_df[iqr_df["protocol"] == protocol].copy()

    # enforce model order
    sub["model_display_name"] = pd.Categorical(
        sub["model_display_name"],
        categories=MODEL_ORDER,
        ordered=True
    )
    sub = sub.sort_values("model_display_name")

    x = np.arange(len(sub))
    width = 0.34

    ax.bar(
        x - width / 2,
        sub["non_outlier"],
        width=width,
        color=BLUE_2[0],
        edgecolor="#2F5D8A",
        linewidth=0.8,
        label="Non-outlier",
        zorder=3
    )
    ax.bar(
        x + width / 2,
        sub["high_iqr_outlier"],
        width=width,
        color=BLUE_2[1],
        edgecolor="#1F4E79",
        linewidth=0.8,
        label="High-IQR outlier",
        zorder=3
    )

    ax.set_title(protocol_title_map[protocol], fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["model_display_name"], rotation=20, ha="right")
    ax.set_ylabel("MAE (kg ha$^{-1}$)", fontsize=11)
    style_axis(ax)

axes[0].legend(frameon=False, fontsize=10, loc="upper left")
fig.suptitle("Figure 6. Outlier sensitivity by validation protocol", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

save_figure(
    fig,
    OUTPUT_DIR / "Figure_6_Outlier_Sensitivity_IQR_Blue.png",
    OUTPUT_DIR / "Figure_6_Outlier_Sensitivity_IQR_Blue.pdf"
)

# =============================================================================
# FIGURE 7
# Yield-tail sensitivity
# =============================================================================
tail_df = pd.read_csv(tail_metrics_file)

# expected columns:
# protocol, model_id, target_yield_tail, mae_kg_ha
required_cols = ["protocol", "model_id", "target_yield_tail", "mae_kg_ha"]
missing_cols = [c for c in required_cols if c not in tail_df.columns]
if missing_cols:
    raise KeyError(f"Figure 7 source is missing columns: {missing_cols}")

tail_df["model_display_name"] = tail_df["model_id"].map(MODEL_NAME_MAP).fillna(tail_df["model_id"])

tail_order = ["bottom_decile", "central_eighty_percent", "top_decile"]
tail_label_map = {
    "bottom_decile": "Bottom decile",
    "central_eighty_percent": "Central 80%",
    "top_decile": "Top decile"
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor="white", sharey=True)

for ax, protocol in zip(axes, protocol_order):
    sub = tail_df[tail_df["protocol"] == protocol].copy()
    sub["model_display_name"] = pd.Categorical(
        sub["model_display_name"],
        categories=MODEL_ORDER,
        ordered=True
    )
    sub["target_yield_tail"] = pd.Categorical(
        sub["target_yield_tail"],
        categories=tail_order,
        ordered=True
    )
    sub = sub.sort_values(["model_display_name", "target_yield_tail"])

    pivot = sub.pivot(index="model_display_name", columns="target_yield_tail", values="mae_kg_ha")
    pivot = pivot.loc[MODEL_ORDER]

    x = np.arange(len(pivot.index))
    width = 0.22

    for i, tail in enumerate(tail_order):
        ax.bar(
            x + (i - 1) * width,
            pivot[tail],
            width=width,
            color=BLUE_3[i],
            edgecolor="#1F4E79",
            linewidth=0.8,
            label=tail_label_map[tail],
            zorder=3
        )

    ax.set_title(protocol_title_map[protocol], fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=20, ha="right")
    ax.set_ylabel("MAE (kg ha$^{-1}$)", fontsize=11)
    style_axis(ax)

axes[0].legend(frameon=False, fontsize=10, loc="upper left")
fig.suptitle("Figure 7. Yield-tail sensitivity by validation protocol", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

save_figure(
    fig,
    OUTPUT_DIR / "Figure_7_Yield_Tail_Sensitivity_Blue.png",
    OUTPUT_DIR / "Figure_7_Yield_Tail_Sensitivity_Blue.pdf"
)

# =============================================================================
# FIGURE 8 and FIGURE 9
# Sequence-length sensitivity
# =============================================================================
seq_df = pd.read_csv(sequence_metrics_file)

# Try to detect the relevant columns automatically
protocol_col = first_existing_column(seq_df, ["protocol"])
model_col = first_existing_column(seq_df, ["model_id", "model"])
seq_len_col = first_existing_column(
    seq_df,
    ["effective_sequence_length", "sequence_length", "history_length", "retained_sequence_length"]
)
mae_col = first_existing_column(
    seq_df,
    ["mae_kg_ha", "mean_mae_kg_ha", "pooled_mae_kg_ha", "mean_pooled_mae_kg_ha"]
)

seq_df["model_display_name"] = seq_df[model_col].map(MODEL_NAME_MAP).fillna(seq_df[model_col])

def make_sequence_plot(df, protocol_value, output_png, output_pdf, figure_title):
    sub = df[df[protocol_col] == protocol_value].copy()

    # keep desired model order if available
    present_models = [m for m in MODEL_ORDER if m in sub["model_display_name"].unique()]

    fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")

    for i, model_name in enumerate(present_models):
        msub = sub[sub["model_display_name"] == model_name].copy()
        msub = msub.sort_values(seq_len_col)

        ax.plot(
            msub[seq_len_col],
            msub[mae_col],
            marker="o",
            markersize=5.8,
            linewidth=2.2,
            color=BLUE_5[i],
            label=model_name.replace("\n", " "),
            zorder=4
        )

    ax.set_xlabel("Effective sequence length (months)", fontsize=11)
    ax.set_ylabel("MAE (kg ha$^{-1}$)", fontsize=11)
    ax.set_title(figure_title, fontsize=13, fontweight="bold")
    ax.set_xticks(sorted(sub[seq_len_col].dropna().unique()))
    style_axis(ax)
    ax.legend(frameon=False, fontsize=9, loc="best")

    plt.tight_layout()

    save_figure(fig, output_png, output_pdf)

make_sequence_plot(
    seq_df,
    "district_grouped",
    OUTPUT_DIR / "Figure_8_Sequence_Length_Sensitivity_District_Grouped_Blue.png",
    OUTPUT_DIR / "Figure_8_Sequence_Length_Sensitivity_District_Grouped_Blue.pdf",
    "Figure 8. Sequence-length sensitivity under district-grouped validation"
)

make_sequence_plot(
    seq_df,
    "rolling_origin",
    OUTPUT_DIR / "Figure_9_Sequence_Length_Sensitivity_Rolling_Origin_Blue.png",
    OUTPUT_DIR / "Figure_9_Sequence_Length_Sensitivity_Rolling_Origin_Blue.pdf",
    "Figure 9. Sequence-length sensitivity under rolling-origin validation"
)

# =============================================================================
# FINAL REPORT
# =============================================================================
print("\nUpdated figures saved to:")
print(OUTPUT_DIR)

for f in sorted(OUTPUT_DIR.glob("*")):
    print(f.name)

Found files:
  IQR contrasts       : D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\outlier_sensitivity\outlier_sensitivity_iqr_contrasts.csv
  Tail metrics        : D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\outlier_sensitivity\outlier_sensitivity_tail_model_metrics.csv
  Sequence metrics    : D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\sequence_length_sensitivity\sequence_length_metric_summary.csv

Updated figures saved to:
D:\MP_Wheat_Project\publication_outputs\updated_blue_figures_6_to_9
Figure_6_Outlier_Sensitivity_IQR_Blue.pdf
Figure_6_Outlier_Sensitivity_IQR_Blue.png
Figure_7_Yield_Tail_Sensitivity_Blue.pdf
Figure_7_Yield_Tail_Sensitivity_Blue.png
Figure_8_Sequence_Length_Sensitivity_District_Grouped_Blue.pdf
Figure_8_Sequence_Length_Sensitivity_District_Grouped_Blue.png
Figure_9_Sequence_Length_Sensitivity_Rolling_Origin_Blue.pdf
Figure_9_Sequence_Length_Sensitivity_Rolling_Origin_Blue.png


In [8]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


# =============================================================================
# INPUT AND OUTPUT PATHS
# =============================================================================

SOURCE_TIF = Path(
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2"
    r"\MP_Wheat_Sentinel2_Publication_Maps"
    r"\mp_wheat_s2_publication_map_source_2022_2023_500m.tif"
)

OUTPUT_DIR = Path(
    r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_10_PNG = OUTPUT_DIR / "Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.png"
FIGURE_10_PDF = OUTPUT_DIR / "Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.pdf"

FIGURE_11_PNG = OUTPUT_DIR / "Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png"
FIGURE_11_PDF = OUTPUT_DIR / "Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.pdf"

FIGURE_12_PNG = OUTPUT_DIR / "Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.png"
FIGURE_12_PDF = OUTPUT_DIR / "Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.pdf"


# =============================================================================
# FILE CHECK
# =============================================================================

if not SOURCE_TIF.exists():
    raise FileNotFoundError(
        f"Source TIFF was not found:\n{SOURCE_TIF}"
    )


# =============================================================================
# READ SCIENTIFIC RASTER BANDS
# Band 2 = peak-growth NDVI
# Band 4 = early-to-peak NDVI change
# Band 5 = peak-to-late NDVI change
# =============================================================================

with rasterio.open(SOURCE_TIF) as src:

    if src.count < 5:
        raise ValueError(
            f"The source TIFF contains only {src.count} bands. "
            "At least five bands are required."
        )

    peak_ndvi = src.read(2, masked=True).astype(np.float64)
    early_to_peak = src.read(4, masked=True).astype(np.float64)
    peak_to_late = src.read(5, masked=True).astype(np.float64)

    bounds = src.bounds
    crs = src.crs
    descriptions = src.descriptions


# Mask NaN and infinite values
peak_ndvi = np.ma.masked_invalid(peak_ndvi)
early_to_peak = np.ma.masked_invalid(early_to_peak)
peak_to_late = np.ma.masked_invalid(peak_to_late)

# Restrict values only to the physically possible NDVI and NDVI-difference ranges
peak_ndvi = np.ma.masked_where(
    (peak_ndvi < -1.0) | (peak_ndvi > 1.0),
    peak_ndvi
)

early_to_peak = np.ma.masked_where(
    (early_to_peak < -2.0) | (early_to_peak > 2.0),
    early_to_peak
)

peak_to_late = np.ma.masked_where(
    (peak_to_late < -2.0) | (peak_to_late > 2.0),
    peak_to_late
)


# =============================================================================
# MAP EXTENT
# =============================================================================

extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top
]


# =============================================================================
# BLUE-BASED COLOUR MAPS
# =============================================================================

# Sequential blue scale for peak-growth NDVI
peak_cmap = plt.get_cmap("Blues").copy()
peak_cmap.set_bad("white")

# Blue-only diverging scale:
# negative change = dark navy
# zero change = white
# positive change = brighter blue
change_cmap = LinearSegmentedColormap.from_list(
    "blue_change",
    [
        (0.00, "#08306B"),
        (0.25, "#4292C6"),
        (0.50, "#F7FBFF"),
        (0.75, "#9ECAE1"),
        (1.00, "#2171B5")
    ]
)

change_cmap.set_bad("white")


# =============================================================================
# COMMON ZERO-CENTRED SCALE FOR FIGURES 11 AND 12
# =============================================================================

combined_change_values = np.concatenate(
    [
        early_to_peak.compressed(),
        peak_to_late.compressed()
    ]
)

if combined_change_values.size == 0:
    raise ValueError(
        "No valid change values were found in Bands 4 and 5."
    )

lower_percentile = np.percentile(combined_change_values, 2)
upper_percentile = np.percentile(combined_change_values, 98)

change_limit = max(
    abs(lower_percentile),
    abs(upper_percentile)
)

if change_limit == 0:
    change_limit = 1.0

change_norm = TwoSlopeNorm(
    vmin=-change_limit,
    vcenter=0.0,
    vmax=change_limit
)


# =============================================================================
# COMMON FIGURE FORMATTING
# =============================================================================

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.linewidth": 0.8,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "savefig.facecolor": "white"
    }
)


def format_map_axes(ax):
    """Apply consistent map-axis formatting without a title or caption."""

    ax.set_xlabel(
        "Longitude (°E)",
        fontsize=11
    )

    ax.set_ylabel(
        "Latitude (°N)",
        fontsize=11
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=9,
        length=4,
        width=0.8
    )

    ax.set_xlim(
        bounds.left,
        bounds.right
    )

    ax.set_ylim(
        bounds.bottom,
        bounds.top
    )

    # Coordinate grid is drawn lightly over the raster so it remains visible.
    ax.grid(
        visible=True,
        which="major",
        axis="both",
        linestyle="--",
        linewidth=0.35,
        color="#345B7E",
        alpha=0.25,
        zorder=3
    )

    for spine in ax.spines.values():
        spine.set_color("#345B7E")
        spine.set_linewidth(0.8)


def save_outputs(fig, png_path, pdf_path):
    """Save high-resolution PNG and vector PDF files."""

    fig.savefig(
        png_path,
        dpi=1000,
        bbox_inches="tight",
        pad_inches=0.04,
        facecolor="white",
        transparent=False
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.04,
        facecolor="white",
        transparent=False
    )

    plt.close(fig)

    print("Saved PNG:", png_path)
    print("Saved PDF:", pdf_path)


# =============================================================================
# FIGURE 10: PEAK-GROWTH NDVI
# No embedded title or caption
# =============================================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5),
    facecolor="white"
)

image = ax.imshow(
    peak_ndvi,
    extent=extent,
    origin="upper",
    cmap=peak_cmap,
    vmin=0.0,
    vmax=0.9,
    interpolation="nearest",
    zorder=2
)

format_map_axes(ax)

colorbar = fig.colorbar(
    image,
    ax=ax,
    orientation="vertical",
    fraction=0.038,
    pad=0.025,
    shrink=0.95
)

colorbar.set_label(
    "Peak-growth NDVI",
    fontsize=11
)

colorbar.ax.tick_params(
    labelsize=9,
    length=3
)

fig.tight_layout()

save_outputs(
    fig,
    FIGURE_10_PNG,
    FIGURE_10_PDF
)


# =============================================================================
# FIGURE 11: EARLY-TO-PEAK NDVI CHANGE
# No embedded title or caption
# =============================================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5),
    facecolor="white"
)

image = ax.imshow(
    early_to_peak,
    extent=extent,
    origin="upper",
    cmap=change_cmap,
    norm=change_norm,
    interpolation="nearest",
    zorder=2
)

format_map_axes(ax)

colorbar = fig.colorbar(
    image,
    ax=ax,
    orientation="vertical",
    fraction=0.038,
    pad=0.025,
    shrink=0.95
)

colorbar.set_label(
    "NDVI change (peak − early)",
    fontsize=11
)

colorbar.ax.tick_params(
    labelsize=9,
    length=3
)

fig.tight_layout()

save_outputs(
    fig,
    FIGURE_11_PNG,
    FIGURE_11_PDF
)


# =============================================================================
# FIGURE 12: PEAK-TO-LATE NDVI CHANGE
# No embedded title or caption
# =============================================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5),
    facecolor="white"
)

image = ax.imshow(
    peak_to_late,
    extent=extent,
    origin="upper",
    cmap=change_cmap,
    norm=change_norm,
    interpolation="nearest",
    zorder=2
)

format_map_axes(ax)

colorbar = fig.colorbar(
    image,
    ax=ax,
    orientation="vertical",
    fraction=0.038,
    pad=0.025,
    shrink=0.95
)

colorbar.set_label(
    "NDVI change (late − peak)",
    fontsize=11
)

colorbar.ax.tick_params(
    labelsize=9,
    length=3
)

fig.tight_layout()

save_outputs(
    fig,
    FIGURE_12_PNG,
    FIGURE_12_PDF
)


# =============================================================================
# EXECUTION SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("FIGURES 10–12 GENERATED SUCCESSFULLY")
print("=" * 90)
print("Source TIFF:", SOURCE_TIF)
print("CRS:", crs)
print("Band descriptions:", descriptions)
print("Common change-map limit:", change_limit)

print("\nPNG outputs:")
print(FIGURE_10_PNG)
print(FIGURE_11_PNG)
print(FIGURE_12_PNG)

print("\nPDF outputs:")
print(FIGURE_10_PDF)
print(FIGURE_11_PDF)
print(FIGURE_12_PDF)

Saved PNG: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.png
Saved PDF: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.pdf
Saved PNG: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png
Saved PDF: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.pdf
Saved PNG: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.png
Saved PDF: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption.pdf

FIGURES 10–12 GENERATED SUCCESSFULLY
Source TIFF: D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_publication_map_source_2022_2023_500m.tif
CRS: EPSG:4326
Band descript

In [9]:
# =============================================================================
# FIGURES 10–12: EXACT TIFF COLOURS, NO EMBEDDED TITLES OR CAPTIONS
# =============================================================================
# Required packages:
# pip install rasterio matplotlib numpy

from pathlib import Path
import math

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects


# =============================================================================
# PROJECT AND OUTPUT DIRECTORIES
# =============================================================================

PROJECT_ROOT = Path(r"D:\MP_Wheat_Project")

SOURCE_DIR = (
    PROJECT_ROOT
    / "publication_outputs"
    / "pubmaps_zip2"
    / "MP_Wheat_Sentinel2_Publication_Maps"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "publication_outputs"
    / "converted_png_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# SOURCE TIFF FILES
# =============================================================================

FIGURE_FILES = {
    10: {
        "filename": "mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif",
        "output_name": "Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption",
    },
    11: {
        "filename": "mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif",
        "output_name": "Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption",
    },
    12: {
        "filename": "mp_wheat_s2_peak_to_late_ndvi_change_2022_2023_500m.tif",
        "output_name": "Figure_12_Peak_to_Late_NDVI_Change_2022_2023_NoCaption",
    },
}


# =============================================================================
# FIND FILE SAFELY
# =============================================================================

def resolve_tiff(filename: str) -> Path:
    """
    Use the expected directory first.
    If the file is not there, search the complete project directory.
    """
    expected_path = SOURCE_DIR / filename

    if expected_path.exists():
        return expected_path

    matches = list(PROJECT_ROOT.rglob(filename))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nCould not find:\n{filename}\n"
            f"Expected directory:\n{SOURCE_DIR}"
        )

    if len(matches) > 1:
        print(f"Multiple copies found for {filename}.")
        print(f"Using: {matches[0]}")

    return matches[0]


# =============================================================================
# READ THE ORIGINAL RGB TIFF WITHOUT CHANGING COLOURS
# =============================================================================

def read_original_rgb(tiff_path: Path):
    """
    Reads the first three TIFF bands exactly as RGB.
    No normalization, stretching, pseudocolor, or rescaling is applied.
    """
    with rasterio.open(tiff_path) as src:

        if src.count < 3:
            raise ValueError(
                f"{tiff_path.name} contains only {src.count} band(s). "
                "A three-band RGB TIFF is required."
            )

        rgb = src.read([1, 2, 3])
        bounds = src.bounds
        crs = src.crs
        transform = src.transform
        descriptions = src.descriptions
        dtype = src.dtypes[0]

    rgb = np.moveaxis(rgb, 0, -1)

    # These publication TIFFs should already be uint8.
    # No scaling is performed.
    if rgb.dtype != np.uint8:
        print(
            f"Warning: {tiff_path.name} has dtype {rgb.dtype}. "
            "Values will only be clipped to the standard RGB range."
        )
        rgb = np.clip(rgb, 0, 255).astype(np.uint8)

    return rgb, bounds, crs, transform, descriptions, dtype


# =============================================================================
# ADD NORTH ARROW
# =============================================================================

def add_north_arrow(ax):
    """
    Adds a small north arrow without adding a map title or caption.
    """
    ax.annotate(
        "N",
        xy=(0.925, 0.925),
        xytext=(0.925, 0.805),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        color="black",
        arrowprops=dict(
            arrowstyle="-|>",
            color="black",
            linewidth=1.8,
            mutation_scale=16,
        ),
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=0.6,
            alpha=0.82,
        ),
        zorder=10,
    )


# =============================================================================
# ADD APPROXIMATE 100-KM SCALE BAR FOR EPSG:4326
# =============================================================================

def add_scale_bar(ax, bounds, scale_length_km=100):
    """
    Adds an approximate horizontal scale bar using the latitude at the
    centre of the displayed map. Appropriate for manuscript visualization.
    """
    middle_latitude = (bounds.bottom + bounds.top) / 2.0

    kilometres_per_degree_longitude = (
        111.32 * math.cos(math.radians(middle_latitude))
    )

    if kilometres_per_degree_longitude <= 0:
        return

    scale_length_degrees = (
        scale_length_km / kilometres_per_degree_longitude
    )

    map_width = bounds.right - bounds.left
    map_height = bounds.top - bounds.bottom

    x_start = bounds.left + (0.075 * map_width)
    x_end = x_start + scale_length_degrees
    y_position = bounds.bottom + (0.075 * map_height)

    # White underlay keeps the scale bar visible on any raster colour.
    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="white",
        linewidth=6.0,
        solid_capstyle="butt",
        zorder=8,
    )

    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="black",
        linewidth=3.0,
        solid_capstyle="butt",
        zorder=9,
    )

    tick_height = 0.015 * map_height

    for x_position in [x_start, x_end]:
        ax.plot(
            [x_position, x_position],
            [
                y_position - tick_height,
                y_position + tick_height,
            ],
            color="black",
            linewidth=1.7,
            zorder=9,
        )

    scale_text = ax.text(
        (x_start + x_end) / 2.0,
        y_position + (0.025 * map_height),
        f"{scale_length_km} km",
        ha="center",
        va="bottom",
        fontsize=9.5,
        color="black",
        fontweight="bold",
        zorder=10,
    )

    scale_text.set_path_effects(
        [
            path_effects.Stroke(
                linewidth=3.0,
                foreground="white",
            ),
            path_effects.Normal(),
        ]
    )


# =============================================================================
# CREATE ONE PUBLICATION MAP
# =============================================================================

def export_rgb_map(
    tiff_path: Path,
    output_stem: str,
    dpi: int = 1000,
):
    """
    Exports one georeferenced RGB TIFF as PNG and PDF.

    The TIFF's RGB values are displayed exactly.
    No title or figure caption is embedded.
    """
    (
        rgb,
        bounds,
        crs,
        transform,
        descriptions,
        dtype,
    ) = read_original_rgb(tiff_path)

    output_png = OUTPUT_DIR / f"{output_stem}.png"
    output_pdf = OUTPUT_DIR / f"{output_stem}.pdf"

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    fig, ax = plt.subplots(
        figsize=(10, 7),
        facecolor="white",
    )

    ax.imshow(
        rgb,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        zorder=2,
    )

    # No title and no caption inside the figure.
    ax.set_title("")

    ax.set_xlabel(
        "Longitude (°E)",
        fontsize=11,
        fontweight="normal",
    )

    ax.set_ylabel(
        "Latitude (°N)",
        fontsize=11,
        fontweight="normal",
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=9.5,
        direction="out",
        length=4,
        width=0.8,
    )

    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)

    # Light coordinate grid; kept subtle to avoid obscuring the TIFF.
    ax.set_axisbelow(False)

    ax.grid(
        True,
        which="major",
        linestyle="--",
        linewidth=0.35,
        color="white",
        alpha=0.35,
        zorder=4,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    add_north_arrow(ax)
    add_scale_bar(
        ax=ax,
        bounds=bounds,
        scale_length_km=100,
    )

    fig.tight_layout()

    fig.savefig(
        output_png,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False,
    )

    fig.savefig(
        output_pdf,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False,
    )

    plt.close(fig)

    print("=" * 90)
    print(f"Source TIFF : {tiff_path}")
    print(f"CRS         : {crs}")
    print(f"Dimensions  : {rgb.shape[1]} × {rgb.shape[0]} pixels")
    print(f"Data type   : {dtype}")
    print(f"Band names  : {descriptions}")
    print(f"PNG saved   : {output_png}")
    print(f"PDF saved   : {output_pdf}")

    return output_png, output_pdf


# =============================================================================
# GENERATE FIGURES 10, 11, AND 12
# =============================================================================

generated_files = []

for figure_number, configuration in FIGURE_FILES.items():

    source_path = resolve_tiff(
        configuration["filename"]
    )

    png_path, pdf_path = export_rgb_map(
        tiff_path=source_path,
        output_stem=configuration["output_name"],
        dpi=1000,
    )

    generated_files.extend(
        [png_path, pdf_path]
    )


# =============================================================================
# FINAL VERIFICATION
# =============================================================================

print("\n" + "=" * 90)
print("FIGURES 10–12 COMPLETED")
print("=" * 90)

for generated_file in generated_files:
    status = "EXISTS" if generated_file.exists() else "MISSING"
    size_mb = (
        generated_file.stat().st_size / (1024 ** 2)
        if generated_file.exists()
        else 0
    )

    print(
        f"{status:7s} | {size_mb:8.2f} MB | {generated_file}"
    )

Source TIFF : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif
CRS         : EPSG:4326
Dimensions  : 1957 × 1292 pixels
Data type   : uint8
Band names  : ('vis-red', 'vis-green', 'vis-blue')
PNG saved   : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.png
PDF saved   : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_NoCaption.pdf
Source TIFF : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif
CRS         : EPSG:4326
Dimensions  : 1957 × 1292 pixels
Data type   : uint8
Band names  : ('vis-red', 'vis-green', 'vis-blue')
PNG saved   : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_NoCaption.png
PDF saved   : D:\MP_Wheat_Project\publicati

In [10]:
# =============================================================================
# UPDATED FIGURES 6–9
#
# Correct numbering:
# Figure 6 = sequence-length sensitivity, district-grouped validation
# Figure 7 = sequence-length sensitivity, rolling-origin validation
# Figure 8 = Tukey high-yield outlier sensitivity
# Figure 9 = lower, central, and upper yield-range sensitivity
#
# Visual requirements:
# - blue shades only
# - dark, clearly distinguishable lines
# - background grid behind lines/bars
# - no manuscript caption embedded inside figure
# - 1000-DPI PNG
# - vector PDF
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


# =============================================================================
# PATHS
# =============================================================================

PROJECT_ROOT = Path(r"D:\MP_Wheat_Project")

SEQUENCE_DIR = (
    PROJECT_ROOT
    / "mp_wheat_project_pipeline"
    / "reporting"
    / "robustness"
    / "sequence_length_sensitivity"
)

OUTLIER_DIR = (
    PROJECT_ROOT
    / "mp_wheat_project_pipeline"
    / "reporting"
    / "robustness"
    / "outlier_sensitivity"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "publication_outputs"
    / "updated_figures_6_to_9"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# OUTPUT SETTINGS
# =============================================================================

PNG_DPI = 1000

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9.5,
    "axes.linewidth": 0.8,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})


# =============================================================================
# BLUE-ONLY PALETTE
# =============================================================================

# Dark, clearly distinguishable line colours
LSTM_BLUE = "#0B3C6F"
RNN_BLUE = "#2878B5"

# Blue-only bar colours; none are excessively pale
BLUE_COMPLETE = "#3D78B2"
BLUE_NON_OUTLIER = "#1F5A94"
BLUE_HIGH_OUTLIER = "#0B2E59"

BLUE_LOWER = "#4D88C7"
BLUE_CENTRAL = "#2369A2"
BLUE_UPPER = "#0B3768"

BAR_EDGE = "#082C50"
GRID_BLUE = "#A9C4DD"
AXIS_BLUE = "#355C7D"


# =============================================================================
# MODEL ORDER AND LABELS
# =============================================================================

MODEL_ORDER = [
    "Prior 2-year moving average",
    "Elastic Net",
    "FT-Transformer",
    "LSTM",
    "Simple RNN",
]

MODEL_LABELS_SHORT = {
    "Prior 2-year moving average": "Prior 2-year\nmoving average",
    "Elastic Net": "Elastic Net",
    "FT-Transformer": "FT-Transformer",
    "LSTM": "LSTM",
    "Simple RNN": "Simple RNN",
}

MODEL_NAME_MAP = {
    "baseline__prior_2year_moving_average": "Prior 2-year moving average",
    "prior_2year_moving_average": "Prior 2-year moving average",
    "Prior 2-year moving average": "Prior 2-year moving average",
    "Prior two-year moving average": "Prior 2-year moving average",

    "linear__elastic_net": "Elastic Net",
    "elastic_net": "Elastic Net",
    "Elastic Net": "Elastic Net",

    "deep_learning__ft_transformer": "FT-Transformer",
    "ft_transformer": "FT-Transformer",
    "FT-Transformer": "FT-Transformer",

    "deep_learning__lstm": "LSTM",
    "lstm": "LSTM",
    "LSTM": "LSTM",

    "deep_learning__simple_rnn": "Simple RNN",
    "simple_rnn": "Simple RNN",
    "SimpleRNN": "Simple RNN",
    "Simple RNN": "Simple RNN",
}


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def normalise_model_name(value):
    value = str(value).strip()
    return MODEL_NAME_MAP.get(value, value)


def resolve_file(expected_path, filename):
    """
    Return the expected file if available; otherwise search the project.
    """
    if expected_path.exists():
        return expected_path

    matches = list(PROJECT_ROOT.rglob(filename))

    if matches:
        print(f"Expected path not found. Using: {matches[0]}")
        return matches[0]

    return None


def style_axis(ax, grid_axis="both"):
    """
    Journal-style white background with grid behind all plotted elements.
    """
    ax.set_facecolor("white")
    ax.set_axisbelow(True)

    ax.grid(
        visible=True,
        which="major",
        axis=grid_axis,
        linestyle="--",
        linewidth=0.55,
        color=GRID_BLUE,
        alpha=0.65,
        zorder=0,
    )

    ax.grid(
        visible=False,
        which="minor",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color(AXIS_BLUE)
    ax.spines["bottom"].set_color(AXIS_BLUE)

    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.8,
        color=AXIS_BLUE,
    )


def save_figure(fig, png_name, pdf_name):
    png_path = OUTPUT_DIR / png_name
    pdf_path = OUTPUT_DIR / pdf_name

    fig.savefig(
        png_path,
        dpi=PNG_DPI,
        bbox_inches="tight",
        pad_inches=0.08,
        facecolor="white",
        transparent=False,
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.08,
        facecolor="white",
        transparent=False,
    )

    plt.close(fig)

    print(f"PNG saved: {png_path}")
    print(f"PDF saved: {pdf_path}")

    return png_path, pdf_path


# =============================================================================
# LOAD SEQUENCE-LENGTH DATA FOR FIGURES 6 AND 7
# =============================================================================

sequence_file = resolve_file(
    SEQUENCE_DIR / "sequence_length_metric_summary.csv",
    "sequence_length_metric_summary.csv",
)

if sequence_file is None:
    raise FileNotFoundError(
        "sequence_length_metric_summary.csv was not found anywhere under "
        "D:\\MP_Wheat_Project"
    )

sequence_df = pd.read_csv(sequence_file)

required_sequence_columns = {
    "protocol",
    "model_display_name",
    "effective_sequence_length",
    "mae_kg_ha",
}

missing_sequence_columns = (
    required_sequence_columns - set(sequence_df.columns)
)

if missing_sequence_columns:
    raise KeyError(
        "The sequence-length source file is missing these columns: "
        f"{sorted(missing_sequence_columns)}\n"
        f"Available columns: {list(sequence_df.columns)}"
    )

sequence_df["protocol"] = (
    sequence_df["protocol"]
    .astype(str)
    .str.strip()
)

sequence_df["model_display_name"] = (
    sequence_df["model_display_name"]
    .astype(str)
    .str.strip()
    .replace({
        "SimpleRNN": "Simple RNN",
        "simple_rnn": "Simple RNN",
        "lstm": "LSTM",
    })
)

sequence_df["effective_sequence_length"] = pd.to_numeric(
    sequence_df["effective_sequence_length"],
    errors="coerce",
)

sequence_df["mae_kg_ha"] = pd.to_numeric(
    sequence_df["mae_kg_ha"],
    errors="coerce",
)

sequence_df = sequence_df.dropna(
    subset=[
        "protocol",
        "model_display_name",
        "effective_sequence_length",
        "mae_kg_ha",
    ]
)

# Collapse any duplicate summary rows safely.
sequence_df = (
    sequence_df
    .groupby(
        [
            "protocol",
            "model_display_name",
            "effective_sequence_length",
        ],
        as_index=False,
    )["mae_kg_ha"]
    .mean()
)


# =============================================================================
# FUNCTION FOR FIGURES 6 AND 7
# =============================================================================

def create_sequence_figure(
    protocol,
    png_name,
    pdf_name,
):
    data = sequence_df[
        sequence_df["protocol"] == protocol
    ].copy()

    if data.empty:
        raise ValueError(
            f"No sequence-length data found for protocol: {protocol}"
        )

    fig, ax = plt.subplots(
        figsize=(8.4, 5.4),
        facecolor="white",
    )

    line_configurations = {
        "LSTM": {
            "color": LSTM_BLUE,
            "marker": "o",
            "linestyle": "-",
            "linewidth": 2.8,
            "markersize": 7.0,
        },
        "Simple RNN": {
            "color": RNN_BLUE,
            "marker": "s",
            "linestyle": "--",
            "linewidth": 2.8,
            "markersize": 6.7,
        },
    }

    for model_name in ["LSTM", "Simple RNN"]:
        model_data = data[
            data["model_display_name"] == model_name
        ].sort_values("effective_sequence_length")

        if model_data.empty:
            warnings.warn(
                f"No rows found for {model_name} under {protocol}."
            )
            continue

        config = line_configurations[model_name]

        ax.plot(
            model_data["effective_sequence_length"],
            model_data["mae_kg_ha"],
            label=model_name,
            color=config["color"],
            linestyle=config["linestyle"],
            linewidth=config["linewidth"],
            marker=config["marker"],
            markersize=config["markersize"],
            markerfacecolor=config["color"],
            markeredgecolor="white",
            markeredgewidth=1.0,
            zorder=5,
        )

    ax.set_xlabel(
        "Effective sequence length (months)"
    )

    ax.set_ylabel(
        "MAE (kg ha$^{-1}$)"
    )

    ax.set_xticks(
        np.arange(1, 8, 1)
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(nbins=7)
    )

    style_axis(
        ax,
        grid_axis="both",
    )

    legend = ax.legend(
        loc="best",
        frameon=True,
        facecolor="white",
        edgecolor="#8CAAC5",
        framealpha=0.95,
    )

    legend.set_zorder(10)

    # No embedded figure title or manuscript caption.
    fig.tight_layout()

    return save_figure(
        fig,
        png_name,
        pdf_name,
    )


# =============================================================================
# FIGURE 6
# EFFECTIVE SEQUENCE-LENGTH SENSITIVITY:
# DISTRICT-GROUPED VALIDATION
# =============================================================================

figure_6_files = create_sequence_figure(
    protocol="district_grouped",
    png_name=(
        "Figure_6_Sequence_Length_Sensitivity_"
        "District_Grouped_Updated.png"
    ),
    pdf_name=(
        "Figure_6_Sequence_Length_Sensitivity_"
        "District_Grouped_Updated.pdf"
    ),
)


# =============================================================================
# FIGURE 7
# EFFECTIVE SEQUENCE-LENGTH SENSITIVITY:
# ROLLING-ORIGIN VALIDATION
# =============================================================================

figure_7_files = create_sequence_figure(
    protocol="rolling_origin",
    png_name=(
        "Figure_7_Sequence_Length_Sensitivity_"
        "Rolling_Origin_Updated.png"
    ),
    pdf_name=(
        "Figure_7_Sequence_Length_Sensitivity_"
        "Rolling_Origin_Updated.pdf"
    ),
)


# =============================================================================
# VERIFIED OUTLIER DATA FOR FIGURE 8
#
# These values reproduce the final Results table.
# They are defined explicitly so that the figure is not dependent on
# uncertain differences in CSV column naming.
# =============================================================================

iqr_data = pd.DataFrame([
    # District-grouped validation
    ["district_grouped", "Prior 2-year moving average", 312.902, 303.387, 1460.443],
    ["district_grouped", "Elastic Net",                 311.687, 302.908, 1370.448],
    ["district_grouped", "FT-Transformer",              320.752, 311.343, 1455.506],
    ["district_grouped", "LSTM",                        307.407, 296.374, 1637.947],
    ["district_grouped", "Simple RNN",                  304.208, 294.257, 1504.324],

    # Rolling-origin validation
    ["rolling_origin", "Prior 2-year moving average",   370.662, 354.493, 1460.443],
    ["rolling_origin", "Elastic Net",                   389.087, 373.471, 1441.600],
    ["rolling_origin", "FT-Transformer",                367.610, 352.084, 1414.020],
    ["rolling_origin", "LSTM",                          430.873, 413.360, 1611.244],
    ["rolling_origin", "Simple RNN",                    459.834, 443.798, 1540.638],
], columns=[
    "protocol",
    "model",
    "complete_sample_mae",
    "non_outlier_mae",
    "high_yield_outlier_mae",
])


# =============================================================================
# FIGURE 8
# MODEL SENSITIVITY TO HIGH-YIELD OBSERVATIONS
# =============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.8, 5.5),
    sharey=True,
    facecolor="white",
)

protocol_configuration = [
    ("district_grouped", "District-grouped validation"),
    ("rolling_origin", "Rolling-origin validation"),
]

for ax, (protocol, panel_title) in zip(
    axes,
    protocol_configuration,
):
    panel = (
        iqr_data[iqr_data["protocol"] == protocol]
        .set_index("model")
        .loc[MODEL_ORDER]
        .reset_index()
    )

    x = np.arange(len(panel))
    width = 0.245

    ax.bar(
        x - width,
        panel["complete_sample_mae"],
        width=width,
        label="Complete sample",
        color=BLUE_COMPLETE,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="",
        zorder=4,
    )

    ax.bar(
        x,
        panel["non_outlier_mae"],
        width=width,
        label="Non-outlier",
        color=BLUE_NON_OUTLIER,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="//",
        zorder=4,
    )

    ax.bar(
        x + width,
        panel["high_yield_outlier_mae"],
        width=width,
        label="High-yield outlier",
        color=BLUE_HIGH_OUTLIER,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="xx",
        zorder=4,
    )

    ax.set_title(
        panel_title,
        fontsize=11,
        fontweight="bold",
        pad=8,
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            MODEL_LABELS_SHORT[name]
            for name in panel["model"]
        ],
        rotation=24,
        ha="right",
    )

    ax.set_ylabel(
        "MAE (kg ha$^{-1}$)"
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(nbins=7)
    )

    style_axis(
        ax,
        grid_axis="y",
    )

axes[0].legend(
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#8CAAC5",
    framealpha=0.96,
)

# No manuscript caption embedded in the image.
fig.tight_layout()

figure_8_files = save_figure(
    fig,
    "Figure_8_High_Yield_Outlier_Sensitivity_Updated.png",
    "Figure_8_High_Yield_Outlier_Sensitivity_Updated.pdf",
)


# =============================================================================
# VERIFIED YIELD-RANGE DATA FOR FIGURE 9
# =============================================================================

yield_range_data = pd.DataFrame([
    # District-grouped validation
    ["district_grouped", "Prior 2-year moving average", 146.907, 304.583, 545.179],
    ["district_grouped", "Elastic Net",                 154.284, 307.903, 499.238],
    ["district_grouped", "FT-Transformer",              199.478, 312.732, 505.921],
    ["district_grouped", "LSTM",                        240.446, 279.617, 595.782],
    ["district_grouped", "Simple RNN",                  231.537, 278.603, 580.876],

    # Rolling-origin validation
    ["rolling_origin", "Prior 2-year moving average",   237.756, 357.155, 608.531],
    ["rolling_origin", "Elastic Net",                   230.296, 378.101, 633.255],
    ["rolling_origin", "FT-Transformer",                198.350, 345.016, 712.453],
    ["rolling_origin", "LSTM",                          264.881, 396.166, 866.587],
    ["rolling_origin", "Simple RNN",                    268.157, 428.919, 891.768],
], columns=[
    "protocol",
    "model",
    "lower_range_mae",
    "central_range_mae",
    "upper_range_mae",
])


# =============================================================================
# FIGURE 9
# LOWER, CENTRAL, AND UPPER OBSERVED-YIELD RANGES
# =============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.8, 5.5),
    sharey=True,
    facecolor="white",
)

for ax, (protocol, panel_title) in zip(
    axes,
    protocol_configuration,
):
    panel = (
        yield_range_data[
            yield_range_data["protocol"] == protocol
        ]
        .set_index("model")
        .loc[MODEL_ORDER]
        .reset_index()
    )

    x = np.arange(len(panel))
    width = 0.245

    ax.bar(
        x - width,
        panel["lower_range_mae"],
        width=width,
        label="Lower range",
        color=BLUE_LOWER,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="",
        zorder=4,
    )

    ax.bar(
        x,
        panel["central_range_mae"],
        width=width,
        label="Central range",
        color=BLUE_CENTRAL,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="//",
        zorder=4,
    )

    ax.bar(
        x + width,
        panel["upper_range_mae"],
        width=width,
        label="Upper range",
        color=BLUE_UPPER,
        edgecolor=BAR_EDGE,
        linewidth=0.8,
        hatch="xx",
        zorder=4,
    )

    ax.set_title(
        panel_title,
        fontsize=11,
        fontweight="bold",
        pad=8,
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            MODEL_LABELS_SHORT[name]
            for name in panel["model"]
        ],
        rotation=24,
        ha="right",
    )

    ax.set_ylabel(
        "MAE (kg ha$^{-1}$)"
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(nbins=7)
    )

    style_axis(
        ax,
        grid_axis="y",
    )

axes[0].legend(
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#8CAAC5",
    framealpha=0.96,
)

# No manuscript caption embedded in the image.
fig.tight_layout()

figure_9_files = save_figure(
    fig,
    "Figure_9_Yield_Range_Sensitivity_Updated.png",
    "Figure_9_Yield_Range_Sensitivity_Updated.pdf",
)


# =============================================================================
# FINAL OUTPUT CHECK
# =============================================================================

all_generated_files = [
    *figure_6_files,
    *figure_7_files,
    *figure_8_files,
    *figure_9_files,
]

print("\n" + "=" * 100)
print("UPDATED FIGURES 6–9 COMPLETED")
print("=" * 100)
print(f"Output directory: {OUTPUT_DIR}\n")

for file_path in all_generated_files:
    exists = file_path.exists()

    if exists:
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(
            f"EXISTS | {size_mb:8.2f} MB | {file_path}"
        )
    else:
        print(
            f"MISSING | {file_path}"
        )

PNG saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_6_Sequence_Length_Sensitivity_District_Grouped_Updated.png
PDF saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_6_Sequence_Length_Sensitivity_District_Grouped_Updated.pdf
PNG saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_7_Sequence_Length_Sensitivity_Rolling_Origin_Updated.png
PDF saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_7_Sequence_Length_Sensitivity_Rolling_Origin_Updated.pdf
PNG saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_8_High_Yield_Outlier_Sensitivity_Updated.png
PDF saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_8_High_Yield_Outlier_Sensitivity_Updated.pdf
PNG saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9\Figure_9_Yield_Range_Sensitivity_Updated.png
PDF saved: D:\MP_Wheat_Project\publication_outputs\updated_figures_6_to_9

In [13]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# =============================================================================
# 1. OUTPUT SETTINGS
# =============================================================================

EXPORT_DPI = 1000
OVERWRITE_ORIGINALS = False


sequence_output_dir = Path(
    r"D:\MP_Wheat_Project\mp_wheat_project_pipeline"
    r"\reporting\robustness\sequence_length_sensitivity"
)

outlier_output_dir = Path(
    r"D:\MP_Wheat_Project\mp_wheat_project_pipeline"
    r"\reporting\robustness\outlier_sensitivity"
)

sequence_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

outlier_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. MEDIUM AND DARK BLUE PALETTE
# =============================================================================

BLUE_DARKEST = "#082F5B"
BLUE_DARK = "#174F82"
BLUE_MEDIUM = "#2C6FA3"
BLUE_ACCENT = "#4F86B5"

EDGE_COLOUR = "#082744"
GRID_COLOUR = "#8FA6BA"
TEXT_COLOUR = "#1F1F1F"


# =============================================================================
# 3. GLOBAL STYLE
# =============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.labelsize": 11.5,
    "axes.edgecolor": TEXT_COLOUR,
    "axes.linewidth": 0.9,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def configure_axis(ax):
    """Apply consistent publication-style formatting."""

    ax.set_axisbelow(True)

    # Grid is behind bars and lines because zorder = 0.
    ax.grid(
        visible=True,
        which="major",
        axis="y",
        linestyle="--",
        linewidth=0.60,
        color=GRID_COLOUR,
        alpha=0.50,
        zorder=0
    )

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=4,
        width=0.8,
        colors=TEXT_COLOUR
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color(
        TEXT_COLOUR
    )

    ax.spines["bottom"].set_color(
        TEXT_COLOUR
    )


def build_output_paths(
    output_directory,
    original_filename
):
    """Create new output names while preserving original images."""

    original_path = Path(
        original_filename
    )

    if OVERWRITE_ORIGINALS:

        png_path = (
            output_directory
            / original_path.name
        )

        pdf_path = (
            output_directory
            / f"{original_path.stem}.pdf"
        )

    else:

        png_path = (
            output_directory
            / (
                f"{original_path.stem}"
                "_Academic_Blue_1000dpi.png"
            )
        )

        pdf_path = (
            output_directory
            / (
                f"{original_path.stem}"
                "_Academic_Blue.pdf"
            )
        )

    return png_path, pdf_path


def save_figure(
    fig,
    png_path,
    pdf_path
):
    """Save high-resolution PNG and vector PDF."""

    fig.savefig(
        png_path,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False
    )

    plt.close(fig)

    print("PNG saved:", png_path)
    print("PDF saved:", pdf_path)
    print()


# =============================================================================
# 5. FIGURE 6: DISTRICT-GROUPED SEQUENCE LENGTH
# =============================================================================

sequence_lengths = np.arange(
    1,
    8
)

district_grouped_lstm = np.array([
    343.397798,
    360.344805,
    354.986236,
    327.919641,
    309.436163,
    307.329306,
    307.407123
])

district_grouped_simple_rnn = np.array([
    349.702273,
    384.663716,
    407.329476,
    346.429441,
    313.430595,
    308.437832,
    304.207792
])


fig6_png, fig6_pdf = build_output_paths(
    sequence_output_dir,
    "Figure_Sequence_Length_Sensitivity_district_grouped.png"
)

fig, ax = plt.subplots(
    figsize=(8.4, 5.4)
)

configure_axis(ax)

ax.plot(
    sequence_lengths,
    district_grouped_lstm,
    color=BLUE_DARKEST,
    linewidth=2.6,
    marker="o",
    markersize=7,
    markerfacecolor="white",
    markeredgecolor=BLUE_DARKEST,
    markeredgewidth=1.8,
    label="LSTM",
    zorder=4
)

ax.plot(
    sequence_lengths,
    district_grouped_simple_rnn,
    color=BLUE_MEDIUM,
    linewidth=2.6,
    marker="s",
    markersize=6.5,
    markerfacecolor="white",
    markeredgecolor=BLUE_MEDIUM,
    markeredgewidth=1.8,
    label="Simple RNN",
    zorder=4
)

ax.set_xlabel(
    "Effective sequence length (months)"
)

ax.set_ylabel(
    r"MAE (kg ha$^{-1}$)"
)

ax.set_xticks(
    sequence_lengths
)

ax.set_xlim(
    0.75,
    7.25
)

ax.set_ylim(
    285,
    420
)

# Explicit tick values prevent locator errors.
ax.set_yticks(
    np.arange(
        300,
        421,
        20
    )
)

legend = ax.legend(
    loc="upper right",
    frameon=True,
    fancybox=False,
    edgecolor="#6F8294",
    facecolor="white",
    framealpha=0.96
)

legend.get_frame().set_linewidth(
    0.7
)

fig.tight_layout()

save_figure(
    fig,
    fig6_png,
    fig6_pdf
)


# =============================================================================
# 6. FIGURE 7: ROLLING-ORIGIN SEQUENCE LENGTH
# =============================================================================

rolling_origin_lstm = np.array([
    471.257777,
    504.652417,
    499.409263,
    466.084884,
    443.952369,
    434.309356,
    430.872957
])

rolling_origin_simple_rnn = np.array([
    487.599297,
    557.766581,
    610.708084,
    536.048405,
    510.318889,
    496.831265,
    459.834077
])


fig7_png, fig7_pdf = build_output_paths(
    sequence_output_dir,
    "Figure_Sequence_Length_Sensitivity_rolling_origin.png"
)

fig, ax = plt.subplots(
    figsize=(8.4, 5.4)
)

configure_axis(ax)

ax.plot(
    sequence_lengths,
    rolling_origin_lstm,
    color=BLUE_DARKEST,
    linewidth=2.6,
    marker="o",
    markersize=7,
    markerfacecolor="white",
    markeredgecolor=BLUE_DARKEST,
    markeredgewidth=1.8,
    label="LSTM",
    zorder=4
)

ax.plot(
    sequence_lengths,
    rolling_origin_simple_rnn,
    color=BLUE_MEDIUM,
    linewidth=2.6,
    marker="s",
    markersize=6.5,
    markerfacecolor="white",
    markeredgecolor=BLUE_MEDIUM,
    markeredgewidth=1.8,
    label="Simple RNN",
    zorder=4
)

ax.set_xlabel(
    "Effective sequence length (months)"
)

ax.set_ylabel(
    r"MAE (kg ha$^{-1}$)"
)

ax.set_xticks(
    sequence_lengths
)

ax.set_xlim(
    0.75,
    7.25
)

ax.set_ylim(
    410,
    635
)

ax.set_yticks(
    np.arange(
        425,
        626,
        25
    )
)

legend = ax.legend(
    loc="upper right",
    frameon=True,
    fancybox=False,
    edgecolor="#6F8294",
    facecolor="white",
    framealpha=0.96
)

legend.get_frame().set_linewidth(
    0.7
)

fig.tight_layout()

save_figure(
    fig,
    fig7_png,
    fig7_pdf
)


# =============================================================================
# 7. COMMON LABELS FOR FIGURES 8 AND 9
# =============================================================================

models = [
    "Two-year\nmoving average",
    "Elastic Net",
    "FT-Transformer",
    "LSTM",
    "Simple RNN"
]

combined_labels = (
    [
        f"{model}\nDG"
        for model in models
    ]
    +
    [
        f"{model}\nRO"
        for model in models
    ]
)

x_positions = np.arange(
    len(combined_labels)
)

bar_width = 0.245


# =============================================================================
# 8. FIGURE 8: IQR OUTLIER SENSITIVITY
# =============================================================================

district_complete = np.array([
    312.902,
    311.687,
    320.752,
    307.407,
    304.208
])

district_non_outlier = np.array([
    303.387,
    302.908,
    311.343,
    296.374,
    294.257
])

district_high_outlier = np.array([
    1460.443,
    1370.448,
    1455.506,
    1637.947,
    1504.324
])

rolling_complete = np.array([
    370.662,
    389.087,
    367.610,
    430.873,
    459.834
])

rolling_non_outlier = np.array([
    354.493,
    373.471,
    352.084,
    413.360,
    443.798
])

rolling_high_outlier = np.array([
    1460.443,
    1441.600,
    1414.020,
    1611.244,
    1540.638
])


complete_values = np.concatenate([
    district_complete,
    rolling_complete
])

non_outlier_values = np.concatenate([
    district_non_outlier,
    rolling_non_outlier
])

high_outlier_values = np.concatenate([
    district_high_outlier,
    rolling_high_outlier
])


fig8_png, fig8_pdf = build_output_paths(
    outlier_output_dir,
    "Figure_Outlier_Sensitivity_IQR.png"
)

fig, ax = plt.subplots(
    figsize=(11.4, 6.2)
)

configure_axis(ax)

ax.bar(
    x_positions - bar_width,
    complete_values,
    width=bar_width,
    color=BLUE_DARKEST,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    label="Complete sample",
    zorder=3
)

ax.bar(
    x_positions,
    non_outlier_values,
    width=bar_width,
    color=BLUE_DARK,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    hatch="//",
    label="Non-outlier observations",
    zorder=3
)

ax.bar(
    x_positions + bar_width,
    high_outlier_values,
    width=bar_width,
    color=BLUE_ACCENT,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    hatch="..",
    label="High-yield outliers",
    zorder=3
)

ax.set_ylabel(
    r"MAE (kg ha$^{-1}$)"
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    combined_labels,
    rotation=35,
    ha="right"
)

ax.set_ylim(
    0,
    1780
)

ax.set_yticks(
    np.arange(
        0,
        1801,
        200
    )
)

ax.axvline(
    4.5,
    color="#536B80",
    linestyle="-",
    linewidth=0.9,
    alpha=0.70,
    zorder=2
)

ax.text(
    2.0,
    1720,
    "District-grouped validation",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=TEXT_COLOUR
)

ax.text(
    7.0,
    1720,
    "Rolling-origin validation",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=TEXT_COLOUR
)

legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.095),
    ncol=3,
    frameon=True,
    fancybox=False,
    edgecolor="#6F8294",
    facecolor="white",
    framealpha=0.98
)

legend.get_frame().set_linewidth(
    0.7
)

fig.tight_layout()

save_figure(
    fig,
    fig8_png,
    fig8_pdf
)


# =============================================================================
# 9. FIGURE 9: YIELD-RANGE SENSITIVITY
# =============================================================================

district_lower = np.array([
    146.907,
    154.284,
    199.478,
    240.446,
    231.537
])

district_central = np.array([
    304.583,
    307.903,
    312.732,
    279.617,
    278.603
])

district_upper = np.array([
    545.179,
    499.238,
    505.921,
    595.782,
    580.876
])

rolling_lower = np.array([
    237.756,
    230.296,
    198.350,
    264.881,
    268.157
])

rolling_central = np.array([
    357.155,
    378.101,
    345.016,
    396.166,
    428.919
])

rolling_upper = np.array([
    608.531,
    633.255,
    712.453,
    866.587,
    891.768
])


lower_values = np.concatenate([
    district_lower,
    rolling_lower
])

central_values = np.concatenate([
    district_central,
    rolling_central
])

upper_values = np.concatenate([
    district_upper,
    rolling_upper
])


fig9_png, fig9_pdf = build_output_paths(
    outlier_output_dir,
    "Figure_Outlier_Sensitivity_Yield_Tails.png"
)

fig, ax = plt.subplots(
    figsize=(11.4, 6.2)
)

configure_axis(ax)

ax.bar(
    x_positions - bar_width,
    lower_values,
    width=bar_width,
    color=BLUE_DARKEST,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    label="Lower yield range",
    zorder=3
)

ax.bar(
    x_positions,
    central_values,
    width=bar_width,
    color=BLUE_DARK,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    hatch="//",
    label="Central yield range",
    zorder=3
)

ax.bar(
    x_positions + bar_width,
    upper_values,
    width=bar_width,
    color=BLUE_ACCENT,
    edgecolor=EDGE_COLOUR,
    linewidth=0.75,
    hatch="..",
    label="Upper yield range",
    zorder=3
)

ax.set_ylabel(
    r"MAE (kg ha$^{-1}$)"
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    combined_labels,
    rotation=35,
    ha="right"
)

ax.set_ylim(
    0,
    980
)

ax.set_yticks(
    np.arange(
        0,
        1001,
        100
    )
)

ax.axvline(
    4.5,
    color="#536B80",
    linestyle="-",
    linewidth=0.9,
    alpha=0.70,
    zorder=2
)

ax.text(
    2.0,
    950,
    "District-grouped validation",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=TEXT_COLOUR
)

ax.text(
    7.0,
    950,
    "Rolling-origin validation",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=TEXT_COLOUR
)

legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.095),
    ncol=3,
    frameon=True,
    fancybox=False,
    edgecolor="#6F8294",
    facecolor="white",
    framealpha=0.98
)

legend.get_frame().set_linewidth(
    0.7
)

fig.tight_layout()

save_figure(
    fig,
    fig9_png,
    fig9_pdf
)


# =============================================================================
# 10. FINAL REPORT
# =============================================================================

print("=" * 90)
print("ALL UPDATED FIGURES COMPLETED SUCCESSFULLY")
print("=" * 90)

print("Figure 6 PNG:", fig6_png)
print("Figure 6 PDF:", fig6_pdf)
print()

print("Figure 7 PNG:", fig7_png)
print("Figure 7 PDF:", fig7_pdf)
print()

print("Figure 8 PNG:", fig8_png)
print("Figure 8 PDF:", fig8_pdf)
print()

print("Figure 9 PNG:", fig9_png)
print("Figure 9 PDF:", fig9_pdf)

PNG saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\sequence_length_sensitivity\Figure_Sequence_Length_Sensitivity_district_grouped_Academic_Blue_1000dpi.png
PDF saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\sequence_length_sensitivity\Figure_Sequence_Length_Sensitivity_district_grouped_Academic_Blue.pdf

PNG saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\sequence_length_sensitivity\Figure_Sequence_Length_Sensitivity_rolling_origin_Academic_Blue_1000dpi.png
PDF saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\sequence_length_sensitivity\Figure_Sequence_Length_Sensitivity_rolling_origin_Academic_Blue.pdf

PNG saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\outlier_sensitivity\Figure_Outlier_Sensitivity_IQR_Academic_Blue_1000dpi.png
PDF saved: D:\MP_Wheat_Project\mp_wheat_project_pipeline\reporting\robustness\outlier_sensitivity\Figure_Outlier_Sensitivity_IQ

In [14]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import numpy as np
import rasterio


# =============================================================================
# 1. INPUT FILES
# =============================================================================

input_folder = Path(
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2"
    r"\MP_Wheat_Sentinel2_Publication_Maps"
)

figure_files = {
    "Figure_10": (
        input_folder
        / "mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif"
    ),
    "Figure_11": (
        input_folder
        / "mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif"
    ),
    "Figure_12": (
        input_folder
        / "mp_wheat_s2_peak_to_late_ndvi_change_2022_2023_500m.tif"
    ),
}


# =============================================================================
# 2. OUTPUT FILES
# =============================================================================

output_folder = Path(
    r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_names = {
    "Figure_10": (
        "Figure_10_Peak_Growth_NDVI_2022_2023_Original_Colours.png"
    ),
    "Figure_11": (
        "Figure_11_Early_to_Peak_NDVI_Change_"
        "2022_2023_Original_Colours.png"
    ),
    "Figure_12": (
        "Figure_12_Peak_to_Late_NDVI_Change_"
        "2022_2023_Original_Colours.png"
    ),
}


# =============================================================================
# 3. EXPORT SETTINGS
# =============================================================================

EXPORT_DPI = 1000
FIGURE_SIZE = (8.5, 6.0)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.linewidth": 0.9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


# =============================================================================
# 4. SUPPORTING FUNCTIONS
# =============================================================================

def make_json_safe(value):
    """Convert metadata values into JSON-compatible objects."""

    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    return str(value)


def calculate_statistics(array):
    """Calculate band statistics without changing raster values."""

    values = array[np.isfinite(array)]

    if values.size == 0:
        return {
            "valid_pixel_count": 0,
            "minimum": None,
            "maximum": None,
            "mean": None,
            "median": None,
            "standard_deviation": None,
            "percentile_25": None,
            "percentile_75": None,
        }

    return {
        "valid_pixel_count": int(values.size),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "standard_deviation": float(np.std(values)),
        "percentile_25": float(np.percentile(values, 25)),
        "percentile_75": float(np.percentile(values, 75)),
    }


def choose_scale_length(map_width_km):
    """Choose a readable scale-bar length."""

    candidates = [
        10,
        20,
        25,
        50,
        100,
        200,
        250,
        500,
        1000,
    ]

    target = map_width_km * 0.20

    permitted = [
        value
        for value in candidates
        if value <= target
    ]

    if permitted:
        return max(permitted)

    return 10


def add_scale_bar(ax, bounds):
    """Add an approximate kilometre scale bar for EPSG:4326."""

    middle_latitude = (
        bounds.bottom + bounds.top
    ) / 2.0

    km_per_longitude_degree = (
        111.32
        * math.cos(
            math.radians(middle_latitude)
        )
    )

    longitude_width = (
        bounds.right - bounds.left
    )

    map_width_km = (
        longitude_width
        * km_per_longitude_degree
    )

    scale_length_km = choose_scale_length(
        map_width_km
    )

    scale_length_degrees = (
        scale_length_km
        / km_per_longitude_degree
    )

    latitude_height = (
        bounds.top - bounds.bottom
    )

    x_start = (
        bounds.left
        + 0.065 * longitude_width
    )

    x_end = (
        x_start
        + scale_length_degrees
    )

    y_position = (
        bounds.bottom
        + 0.070 * latitude_height
    )

    tick_height = (
        0.012 * latitude_height
    )

    # White backing for visibility.
    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="white",
        linewidth=5.0,
        solid_capstyle="butt",
        zorder=6
    )

    # Main black scale line.
    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="black",
        linewidth=2.3,
        solid_capstyle="butt",
        zorder=7
    )

    for x_value in [x_start, x_end]:
        ax.plot(
            [x_value, x_value],
            [
                y_position - tick_height,
                y_position + tick_height
            ],
            color="black",
            linewidth=1.8,
            zorder=7
        )

    label = ax.text(
        (x_start + x_end) / 2.0,
        y_position + 0.022 * latitude_height,
        f"{scale_length_km} km",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color="black",
        zorder=8
    )

    label.set_path_effects([
        path_effects.withStroke(
            linewidth=3,
            foreground="white"
        )
    ])


def add_north_arrow(ax):
    """Add a north arrow."""

    ax.annotate(
        "N",
        xy=(0.935, 0.91),
        xytext=(0.935, 0.78),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="black",
        arrowprops={
            "facecolor": "black",
            "edgecolor": "white",
            "linewidth": 0.8,
            "width": 4.5,
            "headwidth": 12,
            "headlength": 14,
            "shrink": 0.03,
        },
        bbox={
            "boxstyle": "round,pad=0.17",
            "facecolor": "white",
            "edgecolor": "black",
            "linewidth": 0.6,
            "alpha": 0.82,
        },
        zorder=9
    )


def longitude_label(value):
    direction = "E" if value >= 0 else "W"
    return f"{abs(value):.1f}°{direction}"


def latitude_label(value):
    direction = "N" if value >= 0 else "S"
    return f"{abs(value):.1f}°{direction}"


# =============================================================================
# 5. PROCESS ONE RGB TIFF
# =============================================================================

def process_rgb_tiff(
    figure_id,
    input_path,
    output_path
):
    """Export an RGB TIFF without modifying its colours."""

    if not input_path.exists():
        raise FileNotFoundError(
            f"TIFF file not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        if src.count < 3:
            raise ValueError(
                f"{input_path.name} contains only "
                f"{src.count} band(s)."
            )

        red = src.read(1)
        green = src.read(2)
        blue = src.read(3)

        if red.dtype != np.uint8:
            raise TypeError(
                f"{input_path.name} has data type "
                f"{red.dtype}. The script will not rescale it "
                "because that would change the original colours."
            )

        rgb = np.dstack([
            red,
            green,
            blue
        ])

        bounds = src.bounds
        transform = src.transform
        crs = src.crs
        mask = src.dataset_mask()

        metadata = {
            "figure_id": figure_id,
            "input_file": str(input_path),
            "output_file": str(output_path),
            "driver": src.driver,
            "width_pixels": src.width,
            "height_pixels": src.height,
            "band_count": src.count,
            "data_types": list(src.dtypes),
            "crs": str(crs),
            "bounds": {
                "left": bounds.left,
                "bottom": bounds.bottom,
                "right": bounds.right,
                "top": bounds.top,
            },
            "pixel_resolution": {
                "x_degrees": abs(transform.a),
                "y_degrees": abs(transform.e),
            },
            "transform": list(transform),
            "nodata": src.nodata,
            "band_descriptions": list(src.descriptions),
            "colour_interpretation": [
                str(item)
                for item in src.colorinterp
            ],
            "dataset_tags": src.tags(),
            "valid_mask_pixel_count": int(
                np.count_nonzero(mask)
            ),
            "band_statistics": {
                "red": calculate_statistics(red),
                "green": calculate_statistics(green),
                "blue": calculate_statistics(blue),
            },
            "colour_processing": (
                "Original uint8 RGB values preserved exactly. "
                "No stretching, normalisation, gamma correction, "
                "reclassification, or colormap applied."
            ),
        }

    # -------------------------------------------------------------------------
    # CREATE FIGURE
    # -------------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGURE_SIZE,
        facecolor="white"
    )

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

    ax.imshow(
        rgb,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        zorder=1
    )

    ax.set_xlim(
        bounds.left,
        bounds.right
    )

    ax.set_ylim(
        bounds.bottom,
        bounds.top
    )

    # No internal title or caption.
    ax.set_title("")

    ax.set_xlabel(
        "Longitude",
        fontsize=11,
        labelpad=8
    )

    ax.set_ylabel(
        "Latitude",
        fontsize=11,
        labelpad=8
    )

    # Explicit ticks avoid all MaxNLocator errors.
    longitude_ticks = np.linspace(
        bounds.left,
        bounds.right,
        6
    )

    latitude_ticks = np.linspace(
        bounds.bottom,
        bounds.top,
        6
    )

    ax.set_xticks(
        longitude_ticks
    )

    ax.set_yticks(
        latitude_ticks
    )

    ax.set_xticklabels(
        [
            longitude_label(value)
            for value in longitude_ticks
        ],
        fontsize=9.5
    )

    ax.set_yticklabels(
        [
            latitude_label(value)
            for value in latitude_ticks
        ],
        fontsize=9.5
    )

    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.8
    )

    # Do not draw a grid over the satellite raster.
    ax.grid(False)

    add_scale_bar(
        ax,
        bounds
    )

    add_north_arrow(ax)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)

    fig.tight_layout(
        pad=0.8
    )

    fig.savefig(
        output_path,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.04,
        facecolor="white",
        transparent=False
    )

    plt.close(fig)

    print("=" * 90)
    print(f"{figure_id} COMPLETED")
    print("=" * 90)
    print("Input :", input_path)
    print("Output:", output_path)
    print("CRS   :", metadata["crs"])
    print("Bands :", metadata["band_count"])
    print("Dtypes:", metadata["data_types"])
    print("Original RGB colours preserved: True")
    print()

    return metadata


# =============================================================================
# 6. RUN ALL THREE FIGURES
# =============================================================================

plt.close("all")

all_metadata = []

for figure_id, input_path in figure_files.items():

    output_path = (
        output_folder
        / output_names[figure_id]
    )

    result = process_rgb_tiff(
        figure_id=figure_id,
        input_path=input_path,
        output_path=output_path
    )

    all_metadata.append(result)


# =============================================================================
# 7. SAVE METADATA
# =============================================================================

metadata_path = (
    output_folder
    / "Figures_10_11_12_TIFF_Metadata_Original_Colours.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        make_json_safe(all_metadata),
        file,
        indent=2,
        ensure_ascii=False
    )


print("=" * 90)
print("ALL SATELLITE FIGURES COMPLETED")
print("=" * 90)

for figure_id in figure_files:
    print(
        figure_id,
        ":",
        output_folder
        / output_names[figure_id]
    )

print("Metadata:", metadata_path)

Figure_10 COMPLETED
Input : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif
Output: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_Original_Colours.png
CRS   : EPSG:4326
Bands : 3
Dtypes: ['uint8', 'uint8', 'uint8']
Original RGB colours preserved: True

Figure_11 COMPLETED
Input : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif
Output: D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_11_Early_to_Peak_NDVI_Change_2022_2023_Original_Colours.png
CRS   : EPSG:4326
Bands : 3
Dtypes: ['uint8', 'uint8', 'uint8']
Original RGB colours preserved: True

Figure_12 COMPLETED
Input : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_to_late_ndvi_change_2022_2023_500m.tif
Output: D:\MP_Wheat

In [15]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import numpy as np
import rasterio


# =============================================================================
# 1. INPUT FOLDER AND FILES
# =============================================================================

input_folder = Path(
    r"D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2"
    r"\MP_Wheat_Sentinel2_Publication_Maps"
)

# Scientific five-band raster containing the actual NDVI values.
scientific_tif = (
    input_folder
    / "mp_wheat_s2_publication_map_source_2022_2023_500m.tif"
)

# Rendered RGB rasters containing the original colours.
figure_configuration = {
    "Figure_10": {
        "rgb_tif": (
            input_folder
            / "mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif"
        ),
        "scientific_band": 2,
        "expected_description": "peak_ndvi",
        "colourbar_label": "Peak-growth NDVI",
        "output_name": (
            "Figure_10_Peak_Growth_NDVI_2022_2023_"
            "Original_Colours_With_Scale.png"
        ),
    },

    "Figure_11": {
        "rgb_tif": (
            input_folder
            / "mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif"
        ),
        "scientific_band": 4,
        "expected_description": "early_to_peak_ndvi",
        "colourbar_label": "NDVI change (peak - early)",
        "output_name": (
            "Figure_11_Early_to_Peak_NDVI_Change_2022_2023_"
            "Original_Colours_With_Scale.png"
        ),
    },

    "Figure_12": {
        "rgb_tif": (
            input_folder
            / "mp_wheat_s2_peak_to_late_ndvi_change_2022_2023_500m.tif"
        ),
        "scientific_band": 5,
        "expected_description": "peak_to_late_ndvi",
        "colourbar_label": "NDVI change (late - peak)",
        "output_name": (
            "Figure_12_Peak_to_Late_NDVI_Change_2022_2023_"
            "Original_Colours_With_Scale.png"
        ),
    },
}


# =============================================================================
# 2. OUTPUT FOLDER
# =============================================================================

output_folder = Path(
    r"D:\MP_Wheat_Project\publication_outputs\converted_png_outputs"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. FIGURE SETTINGS
# =============================================================================

EXPORT_DPI = 1000
FIGURE_SIZE = (10.5, 7.2)

# Number of colour samples used to reconstruct the colour scale.
COLOUR_SAMPLE_COUNT = 128

# Robust limits avoid a few extreme pixels distorting the side scale.
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.0

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.linewidth": 0.9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


# =============================================================================
# 4. JSON HELPER
# =============================================================================

def make_json_safe(value):
    """Convert values into JSON-compatible objects."""

    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    return str(value)


# =============================================================================
# 5. COORDINATE LABELS
# =============================================================================

def longitude_label(value):
    direction = "E" if value >= 0 else "W"
    return f"{abs(value):.1f}°{direction}"


def latitude_label(value):
    direction = "N" if value >= 0 else "S"
    return f"{abs(value):.1f}°{direction}"


# =============================================================================
# 6. SCALE-BAR FUNCTIONS
# =============================================================================

def choose_scale_length(map_width_km):
    """Choose a suitable rounded scale-bar length."""

    candidates = [
        10,
        20,
        25,
        50,
        100,
        200,
        250,
        500,
        1000,
    ]

    target_length = map_width_km * 0.20

    valid_lengths = [
        candidate
        for candidate in candidates
        if candidate <= target_length
    ]

    if valid_lengths:
        return max(valid_lengths)

    return 10


def add_scale_bar(ax, bounds):
    """Add an approximate geographic scale bar in kilometres."""

    middle_latitude = (
        bounds.bottom + bounds.top
    ) / 2.0

    kilometres_per_longitude_degree = (
        111.32
        * math.cos(
            math.radians(middle_latitude)
        )
    )

    longitude_width = (
        bounds.right - bounds.left
    )

    latitude_height = (
        bounds.top - bounds.bottom
    )

    map_width_km = (
        longitude_width
        * kilometres_per_longitude_degree
    )

    scale_length_km = choose_scale_length(
        map_width_km
    )

    scale_length_degrees = (
        scale_length_km
        / kilometres_per_longitude_degree
    )

    x_start = (
        bounds.left
        + 0.065 * longitude_width
    )

    x_end = (
        x_start
        + scale_length_degrees
    )

    y_position = (
        bounds.bottom
        + 0.070 * latitude_height
    )

    tick_height = (
        0.012 * latitude_height
    )

    # White backing line.
    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="white",
        linewidth=5.2,
        solid_capstyle="butt",
        zorder=7
    )

    # Main scale line.
    ax.plot(
        [x_start, x_end],
        [y_position, y_position],
        color="black",
        linewidth=2.4,
        solid_capstyle="butt",
        zorder=8
    )

    for x_value in [x_start, x_end]:

        ax.plot(
            [x_value, x_value],
            [
                y_position - tick_height,
                y_position + tick_height
            ],
            color="black",
            linewidth=1.8,
            zorder=8
        )

    label = ax.text(
        (x_start + x_end) / 2.0,
        y_position + 0.022 * latitude_height,
        f"{scale_length_km} km",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color="black",
        zorder=9
    )

    label.set_path_effects([
        path_effects.withStroke(
            linewidth=3,
            foreground="white"
        )
    ])


# =============================================================================
# 7. NORTH ARROW
# =============================================================================

def add_north_arrow(ax):
    """Add a north arrow."""

    ax.annotate(
        "N",
        xy=(0.935, 0.91),
        xytext=(0.935, 0.78),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="black",
        arrowprops={
            "facecolor": "black",
            "edgecolor": "white",
            "linewidth": 0.8,
            "width": 4.5,
            "headwidth": 12,
            "headlength": 14,
            "shrink": 0.03,
        },
        bbox={
            "boxstyle": "round,pad=0.17",
            "facecolor": "white",
            "edgecolor": "black",
            "linewidth": 0.6,
            "alpha": 0.85,
        },
        zorder=10
    )


# =============================================================================
# 8. RECONSTRUCT THE ORIGINAL COLOUR SCALE
# =============================================================================

def derive_colourmap_from_rgb_and_values(
    scientific_array,
    rgb_array,
    valid_mask,
    number_of_colours=128
):
    """
    Reconstruct a continuous numeric colour scale by pairing the scientific
    raster values with the already rendered RGB pixels.

    The displayed map remains unchanged. This reconstructed colour map is
    used only for the labelled side colour scale.
    """

    values = scientific_array[valid_mask].astype(
        np.float64
    )

    colours = rgb_array[valid_mask].astype(
        np.float64
    ) / 255.0

    if values.size < 100:
        raise ValueError(
            "Insufficient valid pixels to reconstruct the colour scale."
        )

    # Remove nonfinite values.
    finite_mask = (
        np.isfinite(values)
        & np.all(
            np.isfinite(colours),
            axis=1
        )
    )

    values = values[finite_mask]
    colours = colours[finite_mask]

    # Establish robust numeric scale limits.
    value_minimum = float(
        np.percentile(
            values,
            LOWER_PERCENTILE
        )
    )

    value_maximum = float(
        np.percentile(
            values,
            UPPER_PERCENTILE
        )
    )

    if value_maximum <= value_minimum:
        raise ValueError(
            "Invalid numeric range for the reconstructed colour scale."
        )

    range_mask = (
        (values >= value_minimum)
        & (values <= value_maximum)
    )

    values = values[range_mask]
    colours = colours[range_mask]

    # Sort value-colour pairs.
    sorting_order = np.argsort(
        values
    )

    sorted_values = values[
        sorting_order
    ]

    sorted_colours = colours[
        sorting_order
    ]

    # Divide sorted pixels into quantile groups.
    split_locations = np.linspace(
        0,
        sorted_values.size,
        number_of_colours + 1,
        dtype=int
    )

    colour_positions = []
    representative_colours = []

    for index in range(
        number_of_colours
    ):

        start = split_locations[index]
        end = split_locations[index + 1]

        if end <= start:
            continue

        section_values = sorted_values[
            start:end
        ]

        section_colours = sorted_colours[
            start:end
        ]

        representative_value = float(
            np.median(
                section_values
            )
        )

        representative_colour = np.median(
            section_colours,
            axis=0
        )

        normalised_position = (
            representative_value
            - value_minimum
        ) / (
            value_maximum
            - value_minimum
        )

        colour_positions.append(
            float(
                np.clip(
                    normalised_position,
                    0.0,
                    1.0
                )
            )
        )

        representative_colours.append(
            tuple(
                np.clip(
                    representative_colour,
                    0.0,
                    1.0
                )
            )
        )

    # Remove duplicate positions.
    unique_positions = []
    unique_colours = []

    for position, colour in zip(
        colour_positions,
        representative_colours
    ):

        if (
            len(unique_positions) == 0
            or position
            > unique_positions[-1]
        ):
            unique_positions.append(
                position
            )

            unique_colours.append(
                colour
            )

    if len(unique_positions) < 2:
        raise ValueError(
            "The reconstructed colour scale contains insufficient variation."
        )

    # Guarantee that the colour map starts at zero and ends at one.
    if unique_positions[0] > 0.0:

        unique_positions.insert(
            0,
            0.0
        )

        unique_colours.insert(
            0,
            unique_colours[0]
        )

    if unique_positions[-1] < 1.0:

        unique_positions.append(
            1.0
        )

        unique_colours.append(
            unique_colours[-1]
        )

    colourmap = LinearSegmentedColormap.from_list(
        "original_tiff_colour_scale",
        list(
            zip(
                unique_positions,
                unique_colours
            )
        ),
        N=256
    )

    return (
        colourmap,
        value_minimum,
        value_maximum
    )


# =============================================================================
# 9. PROCESS ONE FIGURE
# =============================================================================

def process_figure(
    figure_id,
    configuration,
    source_path,
    output_directory
):
    """Create one publication PNG with an original-colour numeric scale."""

    rgb_path = configuration[
        "rgb_tif"
    ]

    source_band_number = configuration[
        "scientific_band"
    ]

    expected_description = configuration[
        "expected_description"
    ]

    colourbar_label = configuration[
        "colourbar_label"
    ]

    output_path = (
        output_directory
        / configuration[
            "output_name"
        ]
    )

    if not rgb_path.exists():
        raise FileNotFoundError(
            f"RGB TIFF not found:\n{rgb_path}"
        )

    if not source_path.exists():
        raise FileNotFoundError(
            f"Scientific TIFF not found:\n{source_path}"
        )

    # -------------------------------------------------------------------------
    # READ ORIGINAL RGB TIFF
    # -------------------------------------------------------------------------

    with rasterio.open(
        rgb_path
    ) as rgb_source:

        if rgb_source.count < 3:
            raise ValueError(
                f"{rgb_path.name} does not contain three RGB bands."
            )

        red = rgb_source.read(1)
        green = rgb_source.read(2)
        blue = rgb_source.read(3)

        if (
            red.dtype != np.uint8
            or green.dtype != np.uint8
            or blue.dtype != np.uint8
        ):
            raise TypeError(
                "The RGB TIFF is not uint8. The script will not rescale "
                "the image because rescaling would change its colours."
            )

        rgb_array = np.dstack([
            red,
            green,
            blue
        ])

        bounds = rgb_source.bounds
        rgb_crs = rgb_source.crs
        rgb_transform = rgb_source.transform
        rgb_shape = (
            rgb_source.height,
            rgb_source.width
        )

    # -------------------------------------------------------------------------
    # READ SCIENTIFIC BAND
    # -------------------------------------------------------------------------

    with rasterio.open(
        source_path
    ) as scientific_source:

        scientific_array = scientific_source.read(
            source_band_number
        ).astype(
            np.float64
        )

        scientific_description = (
            scientific_source.descriptions[
                source_band_number - 1
            ]
        )

        scientific_crs = (
            scientific_source.crs
        )

        scientific_transform = (
            scientific_source.transform
        )

        scientific_shape = (
            scientific_source.height,
            scientific_source.width
        )

    # -------------------------------------------------------------------------
    # ALIGNMENT CHECKS
    # -------------------------------------------------------------------------

    if rgb_shape != scientific_shape:
        raise ValueError(
            f"Raster shape mismatch for {figure_id}: "
            f"RGB={rgb_shape}, scientific={scientific_shape}"
        )

    if rgb_crs != scientific_crs:
        raise ValueError(
            f"CRS mismatch for {figure_id}: "
            f"RGB={rgb_crs}, scientific={scientific_crs}"
        )

    if not np.allclose(
        tuple(rgb_transform),
        tuple(scientific_transform),
        rtol=0,
        atol=1e-10
    ):
        raise ValueError(
            f"Raster transform mismatch for {figure_id}."
        )

    if (
        scientific_description
        and scientific_description
        != expected_description
    ):
        raise ValueError(
            f"Unexpected band description for {figure_id}: "
            f"expected {expected_description}, "
            f"found {scientific_description}"
        )

    # -------------------------------------------------------------------------
    # VALID DATA MASK
    # -------------------------------------------------------------------------

    valid_mask = np.isfinite(
        scientific_array
    )

    # The approximately -1 values in this export represent invalid/background
    # pixels and should not contribute to the reconstructed numeric legend.
    valid_mask &= (
        scientific_array > -0.99
    )

    # Remove completely black RGB background pixels where present.
    valid_mask &= np.any(
        rgb_array > 0,
        axis=2
    )

    valid_values = scientific_array[
        valid_mask
    ]

    if valid_values.size == 0:
        raise ValueError(
            f"No valid scientific pixels were found for {figure_id}."
        )

    # -------------------------------------------------------------------------
    # DERIVE MATCHING COLOUR SCALE
    # -------------------------------------------------------------------------

    reconstructed_cmap, scale_minimum, scale_maximum = (
        derive_colourmap_from_rgb_and_values(
            scientific_array=scientific_array,
            rgb_array=rgb_array,
            valid_mask=valid_mask,
            number_of_colours=COLOUR_SAMPLE_COUNT
        )
    )

    scale_normalisation = Normalize(
        vmin=scale_minimum,
        vmax=scale_maximum,
        clip=True
    )

    # -------------------------------------------------------------------------
    # CREATE MAP
    # -------------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGURE_SIZE,
        facecolor="white"
    )

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]

    # Display the original TIFF colours exactly.
    ax.imshow(
        rgb_array,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        zorder=1
    )

    ax.set_xlim(
        bounds.left,
        bounds.right
    )

    ax.set_ylim(
        bounds.bottom,
        bounds.top
    )

    # No embedded title or manuscript caption.
    ax.set_title("")

    ax.set_xlabel(
        "Longitude",
        fontsize=11,
        labelpad=8
    )

    ax.set_ylabel(
        "Latitude",
        fontsize=11,
        labelpad=8
    )

    longitude_ticks = np.linspace(
        bounds.left,
        bounds.right,
        6
    )

    latitude_ticks = np.linspace(
        bounds.bottom,
        bounds.top,
        6
    )

    ax.set_xticks(
        longitude_ticks
    )

    ax.set_yticks(
        latitude_ticks
    )

    ax.set_xticklabels(
        [
            longitude_label(value)
            for value in longitude_ticks
        ],
        fontsize=9.5
    )

    ax.set_yticklabels(
        [
            latitude_label(value)
            for value in latitude_ticks
        ],
        fontsize=9.5
    )

    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.8
    )

    ax.grid(False)

    add_scale_bar(
        ax,
        bounds
    )

    add_north_arrow(
        ax
    )

    # -------------------------------------------------------------------------
    # NUMERIC SIDE COLOUR SCALE
    # -------------------------------------------------------------------------

    scalar_mappable = ScalarMappable(
        norm=scale_normalisation,
        cmap=reconstructed_cmap
    )

    scalar_mappable.set_array([])

    colourbar = fig.colorbar(
        scalar_mappable,
        ax=ax,
        orientation="vertical",
        fraction=0.040,
        pad=0.028,
        shrink=0.88
    )

    colourbar.set_label(
        colourbar_label,
        fontsize=10.5,
        labelpad=10
    )

    colourbar.ax.tick_params(
        labelsize=9,
        direction="out",
        length=3,
        width=0.7
    )

    # Five clearly labelled scale values.
    colourbar_ticks = np.linspace(
        scale_minimum,
        scale_maximum,
        5
    )

    colourbar.set_ticks(
        colourbar_ticks
    )

    colourbar.set_ticklabels([
        f"{value:.2f}"
        for value in colourbar_ticks
    ])

    colourbar.outline.set_linewidth(
        0.7
    )

    # Map neatline.
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)

    fig.tight_layout(
        pad=0.8
    )

    fig.savefig(
        output_path,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
        transparent=False
    )

    plt.close(fig)

    metadata = {
        "figure_id": figure_id,
        "rgb_input": str(rgb_path),
        "scientific_input": str(source_path),
        "scientific_band": source_band_number,
        "scientific_band_description": scientific_description,
        "output_png": str(output_path),
        "crs": str(scientific_crs),
        "width_pixels": scientific_shape[1],
        "height_pixels": scientific_shape[0],
        "valid_scientific_pixels": int(
            valid_values.size
        ),
        "actual_valid_minimum": float(
            np.min(valid_values)
        ),
        "actual_valid_maximum": float(
            np.max(valid_values)
        ),
        "actual_valid_mean": float(
            np.mean(valid_values)
        ),
        "actual_valid_median": float(
            np.median(valid_values)
        ),
        "colourbar_minimum_percentile": (
            LOWER_PERCENTILE
        ),
        "colourbar_maximum_percentile": (
            UPPER_PERCENTILE
        ),
        "colourbar_minimum": float(
            scale_minimum
        ),
        "colourbar_maximum": float(
            scale_maximum
        ),
        "colourbar_label": (
            colourbar_label
        ),
        "colour_handling": (
            "Original RGB raster displayed without recolouring. "
            "Numeric side scale reconstructed from matched scientific "
            "band and RGB pixel pairs."
        ),
    }

    print("=" * 90)
    print(f"{figure_id} COMPLETED")
    print("=" * 90)
    print("RGB input       :", rgb_path)
    print("Scientific band :", source_band_number)
    print("Description     :", scientific_description)
    print("Actual minimum  :", metadata["actual_valid_minimum"])
    print("Actual maximum  :", metadata["actual_valid_maximum"])
    print("Actual mean     :", metadata["actual_valid_mean"])
    print("Colourbar range :", scale_minimum, "to", scale_maximum)
    print("Output          :", output_path)
    print()

    return metadata


# =============================================================================
# 10. GENERATE FIGURES 10–12
# =============================================================================

plt.close("all")

metadata_records = []

for figure_id, configuration in figure_configuration.items():

    result = process_figure(
        figure_id=figure_id,
        configuration=configuration,
        source_path=scientific_tif,
        output_directory=output_folder
    )

    metadata_records.append(
        result
    )


# =============================================================================
# 11. SAVE METADATA
# =============================================================================

metadata_output = (
    output_folder
    / "Figures_10_11_12_Original_Colours_With_Numeric_Scales.json"
)

with open(
    metadata_output,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        make_json_safe(
            metadata_records
        ),
        file,
        indent=2,
        ensure_ascii=False
    )


# =============================================================================
# 12. FINAL REPORT
# =============================================================================

print("=" * 90)
print("ALL THREE FIGURES COMPLETED SUCCESSFULLY")
print("=" * 90)

for figure_id, configuration in figure_configuration.items():

    print(
        figure_id,
        ":",
        output_folder
        / configuration[
            "output_name"
        ]
    )

print(
    "Metadata:",
    metadata_output
)

Figure_10 COMPLETED
RGB input       : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_peak_growth_ndvi_2022_2023_500m.tif
Scientific band : 2
Description     : peak_ndvi
Actual minimum  : -0.9898924231529236
Actual maximum  : 0.9465672373771667
Actual mean     : 0.5461537399108742
Colourbar range : 0.17872890144586565 to 0.8595982146263123
Output          : D:\MP_Wheat_Project\publication_outputs\converted_png_outputs\Figure_10_Peak_Growth_NDVI_2022_2023_Original_Colours_With_Scale.png

Figure_11 COMPLETED
RGB input       : D:\MP_Wheat_Project\publication_outputs\pubmaps_zip2\MP_Wheat_Sentinel2_Publication_Maps\mp_wheat_s2_early_to_peak_ndvi_change_2022_2023_500m.tif
Scientific band : 4
Description     : early_to_peak_ndvi
Actual minimum  : -0.822824239730835
Actual maximum  : 1.1344298124313354
Actual mean     : 0.1384168102730257
Colourbar range : -0.22340329885482788 to 0.5927973747253414
Output          : D:\MP_Wheat_Project\publ